In [ ]:

# TENDORA — Cell 1: Install Dependencies

%pip install -q \
    pymupdf \
    pdfplumber \
    pytesseract \
    pillow \
    pandas \
    numpy \
    scikit-learn \
    sentence-transformers \
    faiss-cpu \
    gradio \
    plotly \
    python-docx \
    openpyxl

print("TENDORA dependencies installed successfully!")


In [ ]:

# TENDORA — Cell 2: Install Tesseract OCR

!apt-get update -qq
!apt-get install -y -qq tesseract-ocr

import pytesseract

print("Tesseract OCR version:", pytesseract.get_tesseract_version())
print("OCR engine ready!")


In [ ]:

# TENDORA — Cell 3: Environment Verification

import fitz
import pandas as pd
import numpy as np
import sklearn
import gradio as gr
import plotly
import faiss
import pytesseract

print("=" * 45)
print("       TENDORA SYSTEM INITIALIZATION")
print("=" * 45)

print("PDF Processing       : READY")
print("OCR Engine           : READY")
print("Data Processing      : READY")
print("Machine Learning     : READY")
print("Vector Search        : READY")
print("Application UI       : READY")

print("=" * 45)
print("TENDORA FOUNDATION INITIALIZED")
print("=" * 45)


In [ ]:

# TENDORA — Cell 4 (Updated): Upload & Extract Tender Package

import zipfile
import os
from google.colab import files

uploaded = files.upload()

zip_name = next(
    (name for name in uploaded if name.lower().endswith(".zip")),
    None
)

if zip_name is None:
    raise ValueError("Please upload the tender ZIP file.")

EXTRACT_DIR = "/content/TENDORA_Tender_Data"
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as archive:
    archive.extractall(EXTRACT_DIR)

print("TENDORA: Tender package extracted successfully!")

for root, dirs, filenames in os.walk(EXTRACT_DIR):
    for filename in filenames:
        print("📄", filename)

# Select primary tender document
TENDER_PATH = os.path.join(
    EXTRACT_DIR, "Tenderdocuments.pdf"
)

assert os.path.exists(TENDER_PATH), "Main tender PDF not found!"

print("\nPrimary document:", TENDER_PATH)
print("Ready for document processing.")


In [ ]:

# TENDORA — Cell 5: Core Tender Processing Engine

import fitz
import pdfplumber
import pytesseract
import pandas as pd
import re
import io

from PIL import Image
from IPython.display import display

def clean_text(text):
    """Clean extracted PDF text without removing meaningful content."""
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_tender_pdf(pdf_path):
    """
    Extract page-wise text and tables.
    Uses OCR as fallback for scanned or text-poor pages.
    """

    document = fitz.open(pdf_path)
    page_records = []

    with pdfplumber.open(pdf_path) as pdf_tables:

        for page_index, page in enumerate(document):

            page_number = page_index + 1

            # 1. Native PDF text extraction
            extracted_text = clean_text(page.get_text("text"))

            extraction_method = "Native PDF Text"

            # 2. OCR fallback for scanned pages
            if len(extracted_text) < 80:

                pix = page.get_pixmap(
                    dpi=250,
                    alpha=False
                )

                image = Image.open(
                    io.BytesIO(pix.tobytes("png"))
                )

                ocr_text = pytesseract.image_to_string(
                    image,
                    lang="eng"
                )

                if len(ocr_text.strip()) > len(extracted_text):
                    extracted_text = clean_text(ocr_text)
                    extraction_method = "OCR"

            # 3. Extract tables
            extracted_tables = []

            table_page = pdf_tables.pages[page_index]

            for table_index, table in enumerate(
                table_page.extract_tables()
            ):

                if not table:
                    continue

                cleaned_table = [
                    [
                        str(cell).strip() if cell is not None else ""
                        for cell in row
                    ]
                    for row in table
                ]

                if len(cleaned_table) >= 2:
                    extracted_tables.append({
                        "table_number": table_index + 1,
                        "rows": cleaned_table
                    })

            # 4. Store structured page information
            page_records.append({
                "page_number": page_number,
                "text": extracted_text,
                "extraction_method": extraction_method,
                "character_count": len(extracted_text),
                "tables": extracted_tables
            })

            print(
                f"Processed page {page_number}/{len(document)} | "
                f"Method: {extraction_method} | "
                f"Characters: {len(extracted_text)} | "
                f"Tables: {len(extracted_tables)}"
            )

    document.close()

    return page_records


print("TENDORA Tender Processing Engine initialized.")


In [ ]:

# TENDORA — Cell 6: Execute Tender Extraction

tender_pages = extract_tender_pdf(TENDER_PATH)

# Create a page-wise dataframe
tender_df = pd.DataFrame([
    {
        "Page": page["page_number"],
        "Extraction Method": page["extraction_method"],
        "Characters": page["character_count"],
        "Tables": len(page["tables"]),
        "Extracted Text": page["text"]
    }
    for page in tender_pages
])

total_pages = len(tender_pages)
total_characters = sum(
    page["character_count"] for page in tender_pages
)
ocr_pages = sum(
    page["extraction_method"] == "OCR"
    for page in tender_pages
)
total_tables = sum(
    len(page["tables"]) for page in tender_pages
)

print("\n" + "=" * 50)
print("       TENDORA EXTRACTION REPORT")
print("=" * 50)

print("Total Pages       :", total_pages)
print("Total Characters  :", total_characters)
print("OCR Pages         :", ocr_pages)
print("Tables Detected   :", total_tables)

print("=" * 50)

display(
    tender_df[
        ["Page", "Extraction Method", "Characters", "Tables"]
    ]
)


In [ ]:

# TENDORA — Cell 7: Preview Extracted Tender Content

for page in tender_pages[:3]:

    print("\n" + "=" * 65)
    print("PAGE:", page["page_number"])
    print("METHOD:", page["extraction_method"])
    print("=" * 65)

    print(page["text"][:2500])

    if page["tables"]:
        print("\nTABLES FOUND:")

        for table in page["tables"]:
            print("\nTable", table["table_number"])
            display(pd.DataFrame(table["rows"]))



In [ ]:

# TENDORA — Cell 8: Multi-Document Inventory

import os
import pandas as pd

document_inventory = []

for root, dirs, filenames in os.walk(EXTRACT_DIR):
    for filename in filenames:

        path = os.path.join(root, filename)
        extension = os.path.splitext(filename)[1].lower()

        if extension not in [".pdf", ".xls", ".xlsx"]:
            continue

        document_inventory.append({
            "Document Name": filename,
            "Document Type": extension.upper(),
            "Size (KB)": round(os.path.getsize(path) / 1024, 2),
            "File Path": path
        })

inventory_df = pd.DataFrame(document_inventory)

print("TENDORA — DOCUMENT INVENTORY")
print("=" * 55)
print("Total documents:", len(inventory_df))

display(inventory_df)

# Save inventory
inventory_df.to_csv(
    "/content/tendora_document_inventory.csv",
    index=False
)

print("Inventory saved successfully!")


In [ ]:

# TENDORA — Cell 9: Excel Compatibility

%pip install -q xlrd

print("Legacy Excel (.xls) support installed!")


In [ ]:

# TENDORA — Cell 10: Unified Multi-Document Processor

import os
import json
import pandas as pd

from collections import defaultdict

unified_documents = []
processing_errors = []

for root, dirs, filenames in os.walk(EXTRACT_DIR):

    for filename in filenames:

        file_path = os.path.join(root, filename)
        extension = os.path.splitext(filename)[1].lower()

        print(f"\nProcessing: {filename}")

        try:

            # --------------------------------
            # PDF PROCESSING
            # --------------------------------

            if extension == ".pdf":

                pages = extract_tender_pdf(file_path)

                for page in pages:

                    unified_documents.append({
                        "source_file": filename,
                        "document_type": "PDF",
                        "section": f"Page {page['page_number']}",
                        "page_number": page["page_number"],
                        "content": page["text"],
                        "tables": page["tables"],
                        "extraction_method": page["extraction_method"]
                    })

                print(f"PDF processed: {len(pages)} pages")

            # --------------------------------
            # EXCEL PROCESSING
            # --------------------------------

            elif extension in [".xls", ".xlsx"]:

                excel_file = pd.ExcelFile(file_path)

                for sheet_name in excel_file.sheet_names:

                    df = pd.read_excel(
                        excel_file,
                        sheet_name=sheet_name,
                        header=None,
                        dtype=str
                    )

                    df = df.fillna("")

                    rows = []

                    for row_index, row in df.iterrows():

                        values = [
                            str(value).strip()
                            for value in row.tolist()
                            if str(value).strip()
                        ]

                        if values:
                            rows.append(
                                f"Row {row_index + 1}: "
                                + " | ".join(values)
                            )

                    sheet_content = "\n".join(rows)

                    unified_documents.append({
                        "source_file": filename,
                        "document_type": "EXCEL",
                        "section": sheet_name,
                        "page_number": None,
                        "content": sheet_content,
                        "tables": [],
                        "extraction_method": "Excel Reader"
                    })

                    print(
                        f"Sheet processed: {sheet_name} "
                        f"({len(df)} rows)"
                    )

        except Exception as e:

            processing_errors.append({
                "file": filename,
                "error": str(e)
            })

            print("ERROR:", str(e))


# --------------------------------
# FINAL DOCUMENT COLLECTION
# --------------------------------

print("\n" + "=" * 60)
print("TENDORA — UNIFIED PROCESSING REPORT")
print("=" * 60)

print("Total document sections:", len(unified_documents))

pdf_sections = sum(
    d["document_type"] == "PDF"
    for d in unified_documents
)

excel_sections = sum(
    d["document_type"] == "EXCEL"
    for d in unified_documents
)

total_characters = sum(
    len(d["content"])
    for d in unified_documents
)

print("PDF pages:", pdf_sections)
print("Excel sheets:", excel_sections)
print("Total extracted characters:", total_characters)
print("Processing errors:", len(processing_errors))

print("=" * 60)

# Save unified collection
with open(
    "/content/tendora_unified_documents.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        unified_documents,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Unified document collection saved!")


In [ ]:

# TENDORA — Cell 11: Extraction Quality Report

quality_report = []

for doc in unified_documents:

    quality_report.append({
        "Source": doc["source_file"],
        "Type": doc["document_type"],
        "Section": doc["section"],
        "Characters": len(doc["content"]),
        "Tables": len(doc["tables"]),
        "Status": (
            "OK" if len(doc["content"].strip()) > 20
            else "CHECK"
        )
    })

quality_df = pd.DataFrame(quality_report)

display(quality_df)

print("\nDOCUMENT-WISE SUMMARY")

display(
    quality_df.groupby("Source").agg(
        Sections=("Section", "count"),
        Total_Characters=("Characters", "sum"),
        Total_Tables=("Tables", "sum")
    )
)

print("\nPROCESSING ERRORS")

if processing_errors:
    display(pd.DataFrame(processing_errors))
else:
    print("No processing errors detected!")


In [ ]:

# TENDORA — Cell 12: BOQ Inspection

boq_docs = [
    doc for doc in unified_documents
    if doc["source_file"] == "BOQ_982557.xls"
]

for doc in boq_docs:

    print("\nSheet:", doc["section"])
    print("-" * 60)
    print(doc["content"][:5000])



In [ ]:

# TENDORA — Cell 13: Structured Excel Extraction

import pandas as pd
import os
import json
from collections import defaultdict

excel_records = []

def column_letter(n):
    """Convert zero-based column index to Excel column name."""
    result = ""
    n += 1

    while n:
        n, remainder = divmod(n - 1, 26)
        result = chr(65 + remainder) + result

    return result


for root, dirs, filenames in os.walk(EXTRACT_DIR):

    for filename in filenames:

        if not filename.lower().endswith((".xls", ".xlsx")):
            continue

        path = os.path.join(root, filename)
        workbook = pd.ExcelFile(path)

        for sheet_name in workbook.sheet_names:

            df = pd.read_excel(
                workbook,
                sheet_name=sheet_name,
                header=None,
                dtype=object
            )

            df = df.where(pd.notna(df), "")

            for row_index, row in df.iterrows():

                cells = {}

                for col_index, value in enumerate(row):

                    if value != "":
                        cells[column_letter(col_index)] = str(value).strip()

                if not cells:
                    continue

                excel_records.append({
                    "source": filename,
                    "sheet": sheet_name,
                    "row": row_index + 1,
                    "cells": cells
                })

print("Structured Excel extraction complete!")
print("Total non-empty rows:", len(excel_records))

# Preview BOQ rows
boq_preview = [
    record for record in excel_records
    if record["source"] == "BOQ_982557.xls"
]

for record in boq_preview[:18]:

    print(
        f"\n{record['sheet']} | Row {record['row']}"
    )

    print(
        json.dumps(
            record["cells"],
            ensure_ascii=False
        )
    )

# Save structured Excel data
with open(
    "/content/tendora_structured_excel.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(excel_records, f, indent=2, ensure_ascii=False)

print("\nStructured Excel data saved!")


In [ ]:

# TENDORA — Cell 14: Tender Intelligence Engine v1

import re
import pandas as pd

# Search patterns for important tender information
PATTERNS = {
    "Tender ID / Reference": [
        r"(?i)(?:tender\s*(?:id|no|number|reference)|nit\s*(?:no|number)|contract\s*no)\s*[:\-]?\s*[A-Z0-9/._-]{5,}"
    ],

    "EMD": [
        r"(?i)(?:earnest\s+money\s+deposit|EMD)\s*[:\-]?\s*.{0,100}"
    ],

    "Tender Fee": [
        r"(?i)(?:tender\s+fee|tender\s+cost|document\s+fee|processing\s+fee)\s*[:\-]?\s*.{0,100}"
    ],

    "Estimated Value": [
        r"(?i)(?:estimated\s+(?:cost|value)|tender\s+value|project\s+cost)\s*[:\-]?\s*.{0,100}"
    ],

    "Submission Deadline": [
        r"(?i).{0,60}(?:last\s+date|due\s+date|deadline|closing\s+date|bid\s+submission).{0,120}"
    ],

    "Opening Date": [
        r"(?i).{0,60}(?:technical\s+bid\s+opening|financial\s+bid\s+opening|bid\s+opening).{0,120}"
    ],

    "Turnover Requirement": [
        r"(?i).{0,70}(?:annual\s+turnover|average\s+annual\s+turnover|financial\s+turnover|average\s+turnover).{0,150}"
    ],

    "Work Experience": [
        r"(?i).{0,70}(?:similar\s+work|similar\s+works|work\s+experience|completed\s+works|experience\s+in\s+execution).{0,150}"
    ],

    "Technical Qualification": [
        r"(?i).{0,70}(?:technical\s+qualification|technical\s+eligibility|technical\s+criteria|qualification\s+criteria).{0,150}"
    ],

    "Required Certificates": [
        r"(?i).{0,70}(?:GST\s+registration|PAN\s+card|EPF|ESIC|registration\s+certificate|valid\s+license).{0,120}"
    ]
}


def extract_candidates(text, source, section, page_number):

    findings = []

    if not text:
        return findings

    lines = text.splitlines()

    for category, patterns in PATTERNS.items():

        for line_number, line in enumerate(lines, start=1):

            for pattern in patterns:

                if re.search(pattern, line):

                    findings.append({
                        "Category": category,
                        "Source": source,
                        "Section": section,
                        "Page": page_number,
                        "Line": line_number,
                        "Evidence": line.strip()
                    })

                    break

    return findings


# Search all PDF pages and Excel sheets
all_findings = []

for doc in unified_documents:

    results = extract_candidates(
        text=doc["content"],
        source=doc["source_file"],
        section=doc["section"],
        page_number=doc["page_number"]
    )

    all_findings.extend(results)


# Convert results into a DataFrame
findings_df = pd.DataFrame(all_findings)

if not findings_df.empty:

    findings_df = findings_df.drop_duplicates(
        subset=["Category", "Source", "Page", "Evidence"]
    ).reset_index(drop=True)

print("=" * 65)
print("       TENDORA — TENDER INTELLIGENCE REPORT")
print("=" * 65)

print("Total candidate findings:", len(findings_df))

if not findings_df.empty:

    display(
        findings_df.groupby("Category")
        .size()
        .reset_index(name="Candidates")
        .sort_values("Candidates", ascending=False)
    )

    print("\nDETECTED INFORMATION")

    for category in PATTERNS:

        category_data = findings_df[
            findings_df["Category"] == category
        ]

        if not category_data.empty:

            print("\n" + "-" * 60)
            print(category.upper())
            print("-" * 60)

            display(
                category_data[
                    ["Source", "Section", "Page", "Evidence"]
                ].head(10)
            )

else:
    print("No matching candidates found. Review document extraction.")

# Save findings
findings_df.to_csv(
    "/content/tendora_intelligence_candidates.csv",
    index=False
)

print("\nCandidate extraction completed!")
print("Results saved to CSV.")


In [ ]:

# TENDORA — Cell 15: Context-Aware Tender Evidence Extraction

import re
import pandas as pd

KEYWORDS = {
    "EMD": [
        "earnest money deposit",
        "emd amount",
        "emd"
    ],
    "Tender Fee": [
        "tender fee",
        "tender cost",
        "document fee"
    ],
    "Estimated Value": [
        "estimated cost of work",
        "estimated cost",
        "estimated value"
    ],
    "Submission Deadline": [
        "last date of submission",
        "last date for submission",
        "deadline for submission of bids",
        "bid submission end date",
        "closing date for submission"
    ],
    "Opening Date": [
        "technical bid opening date",
        "date of opening of bids",
        "opening of technical bids"
    ],
    "Turnover": [
        "gross annual turnover",
        "average annual turnover",
        "minimum turnover",
        "financial turnover"
    ],
    "Work Experience": [
        "similar works completed",
        "similar work experience",
        "completed similar works",
        "experience in execution"
    ],
    "Certificates": [
        "registration certificate",
        "gst registration",
        "pan card",
        "epf",
        "esic"
    ]
}

context_findings = []

for doc in unified_documents:

    text = doc["content"]

    if not text:
        continue

    normalized = text.lower()

    for category, keywords in KEYWORDS.items():

        for keyword in keywords:

            start = 0

            while True:

                position = normalized.find(keyword, start)

                if position == -1:
                    break

                # Extract surrounding context
                left = max(0, position - 350)
                right = min(len(text), position + 650)

                snippet = text[left:right].strip()

                context_findings.append({
                    "Category": category,
                    "Source": doc["source_file"],
                    "Section": doc["section"],
                    "Page": doc["page_number"],
                    "Keyword": keyword,
                    "Context": snippet
                })

                start = position + len(keyword)


# Remove duplicate contexts
context_df = pd.DataFrame(context_findings)

if not context_df.empty:

    context_df = context_df.drop_duplicates(
        subset=["Category", "Source", "Section", "Context"]
    ).reset_index(drop=True)

    print("TOTAL CONTEXTUAL FINDINGS:", len(context_df))

    for category in KEYWORDS:

        subset = context_df[
            context_df["Category"] == category
        ]

        if not subset.empty:

            print("\n" + "=" * 75)
            print(category.upper())
            print("=" * 75)

            for _, row in subset.iterrows():

                print(
                    f"\nSOURCE: {row['Source']} | "
                    f"SECTION: {row['Section']} | "
                    f"PAGE: {row['Page']}"
                )

                print("-" * 60)
                print(row["Context"])

else:
    print("No contextual findings detected.")

# Save contextual evidence
context_df.to_csv(
    "/content/tendora_contextual_evidence.csv",
    index=False
)

print("\nContextual evidence saved successfully!")


In [ ]:

# TENDORA — Cell 16: Verified Tender Profile

import pandas as pd

verified_facts = [
    {
        "Field": "Estimated Project Cost",
        "Value": "₹32,22,988.60 + GST",
        "Source": "ScheduleAtoFUpdated.pdf",
        "Reference": "Page 1",
        "Status": "Found"
    },
    {
        "Field": "Earnest Money Deposit",
        "Value": "₹76,063",
        "Source": "ScheduleAtoFUpdated.pdf",
        "Reference": "Page 1",
        "Status": "Found"
    },
    {
        "Field": "Minimum Bank Solvency",
        "Value": "₹15.21 Lakhs",
        "Source": "NIT1TechnicalBid.xls",
        "Reference": "E-TENDER TECHNICAL BID, Row 30",
        "Status": "Found"
    },
    {
        "Field": "Similar Work Experience",
        "Value": "Details of similar works completed during last 7 years",
        "Source": "NIT1TechnicalBid.xls",
        "Reference": "Form C, Row 4",
        "Status": "Found - Threshold to verify"
    },
    {
        "Field": "Annual Turnover",
        "Value": "Gross annual construction turnover for last 5 years",
        "Source": "NIT1TechnicalBid.xls",
        "Reference": "Form A, Rows 9-12",
        "Status": "Reporting requirement found; minimum not established"
    },
    {
        "Field": "Submission Deadline",
        "Value": "Not established",
        "Source": "Supplied tender package",
        "Reference": "Not located",
        "Status": "Needs Verification"
    },
    {
        "Field": "Tender Fee",
        "Value": "Not established",
        "Source": "Supplied tender package",
        "Reference": "Not located in current findings",
        "Status": "Needs Verification"
    },
    {
        "Field": "Opening Date",
        "Value": "Not established",
        "Source": "Supplied tender package",
        "Reference": "Not located",
        "Status": "Needs Verification"
    }
]

verified_df = pd.DataFrame(verified_facts)

print("=" * 65)
print("       TENDORA — VERIFIED TENDER PROFILE")
print("=" * 65)

display(verified_df)

verified_df.to_csv(
    "/content/tendora_verified_tender_profile.csv",
    index=False
)

print("\nVerified profile generated successfully!")


In [ ]:

# TENDORA — Cell 17: Bidder Company Profile

bidder_profile = {
    "company_name": "Demo Construction Company",

    # Financial details
    "bank_solvency": 1800000,  # INR

    "annual_turnover": {
        "2025-26": None,
        "2024-25": None,
        "2023-24": None,
        "2022-23": None,
        "2021-22": None
    },

    # Previous projects
    "similar_projects": [],

    # Document availability
    "documents": {
        "GST Registration": True,
        "PAN Registration": True,
        "EPF Registration": True,
        "ESIC Registration": False,
        "Work Experience Certificates": True,
        "Performance Certificates": True,
        "Bank Solvency Certificate": True,
        "CA Certified Financial Statements": False,
        "Plant and Machinery List": True,
        "Technical Staff List": True,
        "EMD Proof": True,
        "Local Content Undertaking": True,
        "Non Back-to-Back Work Undertaking": False
    }
}

print("TENDORA — BIDDER PROFILE CREATED")
print("=" * 50)
print("Company:", bidder_profile["company_name"])
print("Bank Solvency: ₹{:,.2f}".format(
    bidder_profile["bank_solvency"]
))
print("Available documents:",
      sum(bidder_profile["documents"].values()))
print("Total document requirements:",
      len(bidder_profile["documents"]))


In [ ]:

# TENDORA — Cell 18: Eligibility Assessment Engine v1

import pandas as pd

assessment = []

def add_result(requirement, result, bidder_value, tender_value, reason):

    assessment.append({
        "Requirement": requirement,
        "Result": result,
        "Bidder Information": bidder_value,
        "Tender Requirement": tender_value,
        "Explanation": reason
    })


# 1. Bank solvency
required_solvency = 1521000
actual_solvency = bidder_profile["bank_solvency"]

if actual_solvency >= required_solvency:
    status = "PASS"
    reason = "Bidder meets the stated bank solvency amount."
else:
    status = "FAIL"
    reason = "Bidder bank solvency is below the stated amount."

add_result(
    "Bank Solvency",
    status,
    f"₹{actual_solvency:,.2f}",
    f"₹{required_solvency:,.2f}",
    reason
)


# 2. Annual turnover
turnover_values = bidder_profile["annual_turnover"]

if any(v is None for v in turnover_values.values()):
    status = "PENDING"
    reason = "Complete five-year turnover figures are unavailable."
else:
    status = "PENDING"
    reason = "Turnover threshold must be verified from the tender."

add_result(
    "Annual Turnover",
    status,
    str(turnover_values),
    "Minimum threshold not established",
    reason
)


# 3. Similar work experience
add_result(
    "Similar Work Experience",
    "PENDING",
    str(len(bidder_profile["similar_projects"])) + " projects provided",
    "Similar completed works during the last 7 years",
    "Qualifying project-value and similarity criteria require verification."
)


# 4. Required documents
for document, available in bidder_profile["documents"].items():

    status = "PASS" if available else "FAIL"

    reason = (
        "Document marked available in bidder profile."
        if available
        else "Document marked unavailable in bidder profile."
    )

    add_result(
        document,
        status,
        "Available" if available else "Missing",
        "Required according to tender checklist",
        reason
    )


# Final assessment
assessment_df = pd.DataFrame(assessment)

print("=" * 65)
print("       TENDORA — BID ELIGIBILITY ASSESSMENT")
print("=" * 65)

display(assessment_df)

print("\nASSESSMENT SUMMARY")

print(assessment_df["Result"].value_counts())

assessment_df.to_csv(
    "/content/tendora_eligibility_assessment.csv",
    index=False
)

print("\nAssessment report saved!")


In [ ]:

# ============================================================
# TENDORA — CELL 19
# COMPLETE DEPENDENCY RECOVERY + DOCUMENT PROCESSING + RAG
# ============================================================

import os
import sys
import json
import zipfile
import subprocess
import importlib.util

import numpy as np
import pandas as pd

print("=" * 70)
print("TENDORA | COMPLETE SEMANTIC SEARCH INITIALIZATION")
print("=" * 70)

# ------------------------------------------------------------
# STEP 1: INSTALL MISSING DEPENDENCIES AUTOMATICALLY
# ------------------------------------------------------------

print("\n[1/9] Checking and repairing dependencies...")

def ensure_package(import_name, package_name=None):
    package_name = package_name or import_name

    try:
        __import__(import_name)
        print(f"  OK: {import_name}")

    except Exception:
        print(f"  Installing: {package_name}")

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            package_name
        ])

        __import__(import_name)
        print(f"  Installed: {import_name}")

ensure_package("fitz", "pymupdf")
ensure_package("pdfplumber")
ensure_package("pytesseract")
ensure_package("PIL", "Pillow")
ensure_package("xlrd")
ensure_package("faiss", "faiss-cpu")
ensure_package("sentence_transformers", "sentence-transformers")

print("Dependency check completed.")

# ------------------------------------------------------------
# STEP 2: IMPORT PROCESSING LIBRARIES
# ------------------------------------------------------------

print("\n[2/9] Importing libraries...")

import fitz
import pdfplumber
import pytesseract
import faiss
import torch

from PIL import Image
from sentence_transformers import SentenceTransformer

print("All imports successful.")

# ------------------------------------------------------------
# STEP 3: UPLOAD AND EXTRACT TENDER ZIP
# ------------------------------------------------------------

print("\n[3/9] Uploading tender ZIP...")

from google.colab import files

EXTRACT_DIR = "/content/TENDORA_Tender_Data"

# Reuse an already extracted folder if it contains documents.
existing_files = []

if os.path.isdir(EXTRACT_DIR):
    for root, dirs, filenames in os.walk(EXTRACT_DIR):
        for filename in filenames:
            if filename.lower().endswith((".pdf", ".xls", ".xlsx")):
                existing_files.append(
                    os.path.join(root, filename)
                )

if existing_files:
    print("Existing tender documents found. Reusing extraction.")

else:
    uploaded = files.upload()

    zip_files = [
        name for name in uploaded
        if name.lower().endswith(".zip")
    ]

    if not zip_files:
        raise ValueError("Upload a valid tender ZIP file.")

    os.makedirs(EXTRACT_DIR, exist_ok=True)

    with zipfile.ZipFile(zip_files[0], "r") as archive:
        archive.extractall(EXTRACT_DIR)

    print("ZIP extraction completed.")

# ------------------------------------------------------------
# STEP 4: INVENTORY DOCUMENTS
# ------------------------------------------------------------

print("\n[4/9] Scanning tender documents...")

document_files = []

for root, dirs, filenames in os.walk(EXTRACT_DIR):
    for filename in filenames:

        if filename.lower().endswith((".pdf", ".xls", ".xlsx")):
            document_files.append(
                os.path.join(root, filename)
            )

if not document_files:
    raise ValueError("No supported tender documents found.")

print("Total documents:", len(document_files))

for path in document_files:
    print("  ✔", os.path.basename(path))

# ------------------------------------------------------------
# STEP 5: PDF EXTRACTION FUNCTION
# ------------------------------------------------------------

print("\n[5/9] Initializing PDF extraction...")

def extract_tender_pdf(pdf_path):

    records = []

    with fitz.open(pdf_path) as pdf:

        for page_index, page in enumerate(pdf):

            page_number = page_index + 1

            text = page.get_text("text").strip()

            extraction_method = "Text Extraction"

            # OCR fallback for scanned pages
            if len(text) < 80:

                pix = page.get_pixmap(
                    matrix=fitz.Matrix(250 / 72, 250 / 72),
                    alpha=False
                )

                image = Image.frombytes(
                    "RGB",
                    [pix.width, pix.height],
                    pix.samples
                )

                text = pytesseract.image_to_string(
                    image,
                    config="--oem 3 --psm 6"
                )

                extraction_method = "OCR"

            # Extract page tables
            tables = []

            try:
                with pdfplumber.open(pdf_path) as pdf_reader:

                    if page_index < len(pdf_reader.pages):
                        tables = (
                            pdf_reader.pages[
                                page_index
                            ].extract_tables() or []
                        )

            except Exception as error:
                print("Table warning:", str(error)[:150])

            records.append({
                "page_number": page_number,
                "text": text,
                "extraction_method": extraction_method,
                "character_count": len(text),
                "tables": tables
            })

    return records

# ------------------------------------------------------------
# STEP 6: PROCESS ALL PDF AND EXCEL DOCUMENTS
# ------------------------------------------------------------

print("\n[6/9] Extracting all tender documents...")

unified_documents = []

for file_path in document_files:

    filename = os.path.basename(file_path)

    extension = os.path.splitext(filename)[1].lower()

    print("\nProcessing:", filename)

    try:

        if extension == ".pdf":

            pages = extract_tender_pdf(file_path)

            for page in pages:

                unified_documents.append({
                    "source_file": filename,
                    "document_type": "PDF",
                    "section": f"Page {page['page_number']}",
                    "page_number": page["page_number"],
                    "content": page["text"],
                    "tables": page["tables"],
                    "extraction_method": page["extraction_method"]
                })

            print("  Pages:", len(pages))

        elif extension in [".xls", ".xlsx"]:

            excel_file = pd.ExcelFile(file_path)

            for sheet_name in excel_file.sheet_names:

                df = pd.read_excel(
                    excel_file,
                    sheet_name=sheet_name,
                    header=None,
                    dtype=str
                )

                lines = []

                for row_index, row in df.iterrows():

                    cells = []

                    for col_index, value in enumerate(row):

                        if pd.notna(value) and str(value).strip():

                            # Excel column name
                            column_number = col_index + 1
                            column_name = ""

                            while column_number:
                                column_number, remainder = divmod(
                                    column_number - 1, 26
                                )
                                column_name = (
                                    chr(65 + remainder) + column_name
                                )

                            cells.append(
                                f"{column_name}: {str(value).strip()}"
                            )

                    if cells:
                        lines.append(
                            f"Row {row_index + 1}: "
                            + " | ".join(cells)
                        )

                sheet_content = "\n".join(lines)

                if sheet_content.strip():

                    unified_documents.append({
                        "source_file": filename,
                        "document_type": "Excel",
                        "section": sheet_name,
                        "page_number": None,
                        "content": sheet_content,
                        "tables": [],
                        "extraction_method": "Structured Excel Extraction"
                    })

                print(
                    "  Sheet:", sheet_name,
                    "| Rows:", len(df)
                )

    except Exception as error:
        print("  ERROR:", filename, str(error))

if not unified_documents:
    raise ValueError("No document content was extracted.")

# Save extracted documents
with open(
    "/content/tendora_unified_documents.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        unified_documents,
        file,
        ensure_ascii=False,
        indent=2
    )

print("\nExtracted sections:", len(unified_documents))

# ------------------------------------------------------------
# STEP 7: LOAD EMBEDDING MODEL
# ------------------------------------------------------------

print("\n[7/9] Loading embedding model...")

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded:", MODEL_NAME)

# ------------------------------------------------------------
# STEP 8: CREATE KNOWLEDGE CHUNKS AND EMBEDDINGS
# ------------------------------------------------------------

print("\n[8/9] Creating knowledge chunks...")

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

knowledge_chunks = []

for doc in unified_documents:

    content = doc.get("content", "")

    if not content or len(content.strip()) < 30:
        continue

    start = 0

    while start < len(content):

        end = start + CHUNK_SIZE

        chunk_text = content[start:end].strip()

        if len(chunk_text) >= 30:

            knowledge_chunks.append({
                "text": chunk_text,
                "source": doc["source_file"],
                "section": doc["section"],
                "page": doc["page_number"],
                "document_type": doc["document_type"]
            })

        start += CHUNK_SIZE - CHUNK_OVERLAP

if not knowledge_chunks:
    raise ValueError("No valid knowledge chunks generated.")

print("Knowledge chunks:", len(knowledge_chunks))

chunk_texts = [
    chunk["text"] for chunk in knowledge_chunks
]

print("Generating embeddings...")

embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.asarray(
    embeddings,
    dtype=np.float32
)

# ------------------------------------------------------------
# STEP 9: BUILD AND SAVE FAISS INDEX
# ------------------------------------------------------------

print("\n[9/9] Building FAISS vector index...")

dimension = embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(dimension)

faiss_index.add(embeddings)

INDEX_PATH = "/content/tendora_faiss.index"

CHUNKS_PATH = "/content/tendora_knowledge_chunks.json"

faiss.write_index(
    faiss_index,
    INDEX_PATH
)

with open(
    CHUNKS_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        knowledge_chunks,
        file,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TENDORA | SEMANTIC SEARCH ENGINE READY")
print("=" * 70)

print("Original Documents :", len(document_files))
print("Processed Sections :", len(unified_documents))
print("Knowledge Chunks   :", len(knowledge_chunks))
print("Embedding Model    :", MODEL_NAME)
print("Vector Dimension   :", dimension)
print("Indexed Vectors    :", faiss_index.ntotal)

print("\nSaved files:")
print(INDEX_PATH)
print(CHUNKS_PATH)
print("/content/tendora_unified_documents.json")

print("\nCELL 19 COMPLETED SUCCESSFULLY.")


In [ ]:

# ============================================================
# TENDORA — CELL 20
# SEMANTIC SEARCH AND EVIDENCE RETRIEVAL
# ============================================================

import numpy as np
import pandas as pd
import faiss

print("=" * 70)
print("TENDORA | SEMANTIC TENDER SEARCH")
print("=" * 70)

# ------------------------------------------------------------
# STEP 1: VALIDATE VECTOR INDEX
# ------------------------------------------------------------

if "faiss_index" not in globals():
    faiss_index = faiss.read_index(
        "/content/tendora_faiss.index"
    )

if "knowledge_chunks" not in globals():
    import json

    with open(
        "/content/tendora_knowledge_chunks.json",
        "r",
        encoding="utf-8"
    ) as file:
        knowledge_chunks = json.load(file)

if "embedding_model" not in globals():
    from sentence_transformers import SentenceTransformer

    embedding_model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )

print("FAISS vectors:", faiss_index.ntotal)
print("Knowledge chunks:", len(knowledge_chunks))

# ------------------------------------------------------------
# STEP 2: DEFINE SEMANTIC SEARCH FUNCTION
# ------------------------------------------------------------

def search_tender(query, top_k=5):

    if not query or not query.strip():
        raise ValueError("Please enter a search question.")

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(
        query_embedding,
        min(top_k, len(knowledge_chunks))
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx < 0:
            continue

        chunk = knowledge_chunks[idx]

        results.append({
            "similarity": round(float(score), 4),
            "source": chunk["source"],
            "section": chunk["section"],
            "page": chunk["page"],
            "document_type": chunk["document_type"],
            "content": chunk["text"]
        })

    return results

# ------------------------------------------------------------
# STEP 3: DISPLAY SEARCH RESULTS
# ------------------------------------------------------------

def display_tender_results(query, top_k=5):

    results = search_tender(query, top_k)

    print("\nQUESTION:", query)
    print("=" * 70)

    for i, result in enumerate(results, 1):

        print(f"\nRESULT {i}")
        print("-" * 70)

        print("Similarity :", result["similarity"])
        print("Source     :", result["source"])
        print("Section    :", result["section"])
        print("Page       :", result["page"])
        print("Type       :", result["document_type"])

        print("\nEvidence:")
        print(result["content"][:1500])

    return results

# ------------------------------------------------------------
# STEP 4: TEST TENDER QUESTIONS
# ------------------------------------------------------------

test_questions = [
    "What is the earnest money deposit amount?",
    "What is the estimated cost of the project?",
    "What is the minimum bank solvency requirement?",
    "What are the similar work experience requirements?",
    "Which certificates must the bidder submit?",
    "What is the technical scope of the work?"
]

all_search_results = {}

for question in test_questions:

    all_search_results[question] = display_tender_results(
        question,
        top_k=5
    )

# ------------------------------------------------------------
# STEP 5: CREATE SEARCH QUALITY SUMMARY
# ------------------------------------------------------------

summary_rows = []

for question, results in all_search_results.items():

    best_result = results[0] if results else {}

    summary_rows.append({
        "Question": question,
        "Top Similarity": best_result.get("similarity"),
        "Top Source": best_result.get("source"),
        "Top Section": best_result.get("section"),
        "Top Page": best_result.get("page")
    })

search_quality_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 70)
print("TENDORA | SEARCH TEST SUMMARY")
print("=" * 70)

display(search_quality_df)

search_quality_df.to_csv(
    "/content/tendora_search_test_summary.csv",
    index=False
)

print("\nSearch summary saved successfully.")
print("CELL 20 COMPLETED.")


In [ ]:

# ============================================================
# TENDORA — CELL 21
# AI-POWERED TENDER QUESTION ANSWERING
# ============================================================

import os
import torch
import numpy as np
import pandas as pd

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("=" * 70)
print("TENDORA | AI TENDER INTELLIGENCE ASSISTANT")
print("=" * 70)

# ------------------------------------------------------------
# STEP 1: VERIFY PREVIOUS COMPONENTS
# ------------------------------------------------------------

print("\n[1/5] Validating semantic search engine...")

required_variables = [
    "faiss_index",
    "knowledge_chunks",
    "embedding_model",
    "search_tender"
]

for variable in required_variables:
    if variable not in globals():
        raise ValueError(
            f"{variable} not found. Complete Cells 19 and 20 first."
        )

print("FAISS index:", faiss_index.ntotal)
print("Knowledge chunks:", len(knowledge_chunks))

# ------------------------------------------------------------
# STEP 2: LOAD GENERATIVE LANGUAGE MODEL
# ------------------------------------------------------------

print("\n[2/5] Loading FLAN-T5 language model...")

MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

generation_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

device = "cuda" if torch.cuda.is_available() else "cpu"

generation_model = generation_model.to(device)
generation_model.eval()

print("Language model loaded:", MODEL_NAME)
print("Execution device:", device)

# ------------------------------------------------------------
# STEP 3: BUILD EVIDENCE CONTEXT
# ------------------------------------------------------------

print("\n[3/5] Initializing evidence retrieval...")

def retrieve_tender_evidence(question, top_k=5):

    results = search_tender(
        question,
        top_k=top_k
    )

    evidence_blocks = []

    for i, result in enumerate(results, 1):

        source = result.get("source", "Unknown")
        section = result.get("section", "Unknown")
        page = result.get("page")

        reference = (
            f"{source}, Page {page}"
            if page is not None
            else f"{source}, Sheet/Section: {section}"
        )

        evidence_blocks.append({
            "evidence_id": i,
            "reference": reference,
            "similarity": result["similarity"],
            "content": result["content"]
        })

    return evidence_blocks

# ------------------------------------------------------------
# STEP 4: GENERATE GROUNDED ANSWER
# ------------------------------------------------------------

print("\n[4/5] Creating answer generation engine...")

def ask_tendora(question, top_k=5):

    if not question or not question.strip():
        raise ValueError("Please enter a question.")

    evidence = retrieve_tender_evidence(
        question,
        top_k=top_k
    )

    if not evidence:
        return {
            "question": question,
            "answer": "I could not find relevant evidence in the uploaded tender documents.",
            "sources": []
        }

    # Build context with source references
    context_parts = []

    for item in evidence:

        context_parts.append(
            f"[SOURCE {item['evidence_id']}: {item['reference']}]\n"
            f"{item['content'][:1800]}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are TENDORA, an AI-powered Indian government tender
intelligence assistant.

Answer the user's question using ONLY the evidence provided below.

STRICT RULES:
1. Do not invent amounts, dates, eligibility criteria or conditions.
2. If the evidence does not establish an answer, say:
   "The available tender evidence is insufficient to confirm this."
3. Distinguish an explicit requirement from a form field or template.
4. Do not interpret blank or zero BOQ placeholders as actual prices.
5. Preserve the exact meaning of tender conditions.
6. If a value is uncertain, clearly mention that it needs verification.
7. Give a concise, structured answer.
8. Do not invent source references.

TENDER EVIDENCE:
{context}

USER QUESTION:
{question}

ANSWER:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(device)

    with torch.no_grad():

        output = generation_model.generate(
            **inputs,
            max_new_tokens=220,
            num_beams=4,
            do_sample=False,
            repetition_penalty=1.15,
            early_stopping=True
        )

    answer = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    # Prepare source references
    sources = []

    for item in evidence:

        sources.append({
            "source": item["reference"],
            "similarity": item["similarity"]
        })

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_evidence": evidence
    }

# ------------------------------------------------------------
# STEP 5: TEST TENDORA AI ASSISTANT
# ------------------------------------------------------------

print("\n[5/5] Testing AI question-answering engine...")

test_questions = [
    "What is the estimated project cost?",
    "How much EMD must the bidder submit?",
    "What is the minimum bank solvency requirement?",
    "What documents should the bidder prepare?"
]

tendora_test_results = []

for question in test_questions:

    print("\n" + "=" * 70)
    print("USER:", question)
    print("-" * 70)

    result = ask_tendora(question)

    print("\nTENDORA:", result["answer"])

    print("\nSOURCE REFERENCES:")

    for source in result["sources"]:
        print(
            "•",
            source["source"],
            "| Similarity:",
            source["similarity"]
        )

    tendora_test_results.append(result)

print("\n" + "=" * 70)
print("TENDORA AI ASSISTANT INITIALIZED")
print("=" * 70)

print("Language Model:", MODEL_NAME)
print("Vector Database: FAISS")
print("Embedding Model: all-MiniLM-L6-v2")
print("Grounded Retrieval: Active")
print("Source References: Active")

print("\nCELL 21 COMPLETED SUCCESSFULLY.")


In [ ]:

# ============================================================
# TENDORA — CELL 22
# HYBRID RETRIEVAL + TENDER EVIDENCE VERIFICATION
# ============================================================

import re
import pandas as pd
import numpy as np

print("=" * 75)
print("TENDORA | TENDER EVIDENCE VERIFICATION ENGINE")
print("=" * 75)

# ------------------------------------------------------------
# STEP 1: VERIFY REQUIRED COMPONENTS
# ------------------------------------------------------------

required = [
    "faiss_index",
    "knowledge_chunks",
    "embedding_model"
]

for name in required:
    if name not in globals():
        raise ValueError(
            f"{name} is missing. Complete Cell 19 first."
        )

print("FAISS vectors:", faiss_index.ntotal)
print("Knowledge chunks:", len(knowledge_chunks))

# ------------------------------------------------------------
# STEP 2: HYBRID RETRIEVAL FUNCTION
# ------------------------------------------------------------

def normalize_text(text):
    return re.sub(
        r"[^a-z0-9₹]+",
        " ",
        str(text).lower()
    ).strip()


def hybrid_tender_search(query, top_k=12):

    query_vector = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Retrieve a wider initial candidate set
    scores, indices = faiss_index.search(
        query_vector,
        min(40, len(knowledge_chunks))
    )

    query_terms = set(normalize_text(query).split())

    results = []

    for similarity, idx in zip(scores[0], indices[0]):

        if idx < 0:
            continue

        chunk = knowledge_chunks[idx]

        content = chunk["text"]

        content_terms = set(normalize_text(content).split())

        matched_terms = query_terms.intersection(content_terms)

        keyword_score = (
            len(matched_terms) / max(len(query_terms), 1)
        )

        # Combined ranking score
        combined_score = (
            0.75 * float(similarity)
            + 0.25 * keyword_score
        )

        results.append({
            "semantic_score": round(float(similarity), 4),
            "keyword_score": round(keyword_score, 4),
            "combined_score": round(combined_score, 4),
            "matched_terms": ", ".join(sorted(matched_terms)),
            "source": chunk["source"],
            "section": chunk["section"],
            "page": chunk["page"],
            "document_type": chunk["document_type"],
            "content": content
        })

    results.sort(
        key=lambda x: x["combined_score"],
        reverse=True
    )

    return results[:top_k]


# ------------------------------------------------------------
# STEP 3: DEFINE IMPORTANT TENDER AUDIT QUESTIONS
# ------------------------------------------------------------

audit_questions = {
    "EMD Requirement":
        "Earnest Money Deposit EMD amount exemption payment security deposit",

    "Bank Solvency":
        "Minimum bank solvency certificate required amount eligibility",

    "Estimated Project Cost":
        "Estimated cost of work project value estimated tender cost GST",

    "Tender Fee":
        "Tender document fee tender fee payment amount",

    "Submission Deadline":
        "Last date and time for submission of online tender bid",

    "Bid Opening":
        "Technical bid opening date and time",

    "Similar Work Experience":
        "Eligibility criteria similar work experience completed projects last seven years",

    "Annual Turnover":
        "Minimum annual construction turnover financial years CA certificate",

    "Required Documents":
        "Documents certificates registrations undertakings bidder must submit technical bid",

    "Technical Scope":
        "Scope of work replacement IBR steam piping fittings steam pressure reducing station"
}


# ------------------------------------------------------------
# STEP 4: RETRIEVE AND DISPLAY EVIDENCE
# ------------------------------------------------------------

print("\nRunning evidence verification...\n")

audit_results = []
audit_evidence = {}

for category, query in audit_questions.items():

    print("\n" + "=" * 75)
    print("CATEGORY:", category)
    print("=" * 75)

    results = hybrid_tender_search(
        query,
        top_k=8
    )

    audit_evidence[category] = results

    for rank, result in enumerate(results, 1):

        reference = (
            f"Page {result['page']}"
            if result["page"] is not None
            else f"Sheet: {result['section']}"
        )

        print(f"\nRESULT {rank}")
        print("-" * 60)

        print("Source:", result["source"])
        print("Location:", reference)
        print("Semantic score:", result["semantic_score"])
        print("Keyword score:", result["keyword_score"])
        print("Combined score:", result["combined_score"])

        print("\nEvidence:")
        print(result["content"][:1800])

        audit_results.append({
            "category": category,
            "rank": rank,
            "source": result["source"],
            "section": result["section"],
            "page": result["page"],
            "semantic_score": result["semantic_score"],
            "keyword_score": result["keyword_score"],
            "combined_score": result["combined_score"],
            "matched_terms": result["matched_terms"],
            "evidence": result["content"]
        })


# ------------------------------------------------------------
# STEP 5: SAVE EVIDENCE AUDIT
# ------------------------------------------------------------

audit_df = pd.DataFrame(audit_results)

AUDIT_PATH = "/content/tendora_evidence_audit.csv"

audit_df.to_csv(
    AUDIT_PATH,
    index=False
)

print("\n" + "=" * 75)
print("TENDORA | EVIDENCE AUDIT COMPLETED")
print("=" * 75)

print("Categories audited:", len(audit_questions))
print("Evidence records:", len(audit_df))
print("Output:", AUDIT_PATH)

print("\nCELL 22 COMPLETED SUCCESSFULLY.")


In [ ]:

# ============================================================
# TENDORA | CELL 23
# Verified Tender Facts + Evidence-First Q&A
# ============================================================

import re
import pandas as pd
from IPython.display import display, Markdown

# 1. Verified facts extracted from the uploaded BARC tender
VERIFIED_TENDER_FACTS = {
    "estimated_cost": {
        "value": "₹32,22,988.60 + GST as applicable",
        "source": "ScheduleAtoFUpdated.pdf, Page 1; Tenderdocuments.pdf, Page 6",
        "keywords": ["estimated cost", "project cost", "tender value", "estimated amount"]
    },
    "emd": {
        "value": "₹76,063",
        "source": "ScheduleAtoFUpdated.pdf, Page 1; Tenderdocuments.pdf, Page 6",
        "keywords": ["emd", "earnest money", "earnest money deposit"]
    },
    "bank_solvency": {
        "value": "Minimum ₹15.21 lakh",
        "source": "NIT1TechnicalBid.xls, E-TENDER TECHNICAL BID, Row 30",
        "keywords": ["bank solvency", "solvency certificate", "minimum solvency"]
    },
    "turnover": {
        "value": "Minimum ₹38.03 lakh; CA-certified turnover for the last 3 years",
        "source": "NIT1TechnicalBid.xls, E-TENDER TECHNICAL BID, Row 19",
        "keywords": ["turnover", "annual turnover", "financial turnover"]
    },
    "security_deposit": {
        "value": "2.5% of the tendered value",
        "source": "Tenderdocuments.pdf, Page 6",
        "keywords": ["security deposit", "security deposit percentage"]
    },
    "performance_guarantee": {
        "value": "3% of the tendered value",
        "source": "ScheduleAtoFUpdated.pdf, Page 2",
        "keywords": ["performance guarantee", "performance security"]
    },
    "bank_guarantee_release": {
        "value": "The ₹5 lakh reference relates to the security-deposit bank-guarantee release condition. It is NOT the bidder's minimum bank-solvency requirement.",
        "source": "Tenderdocuments.pdf, Page 13",
        "keywords": ["5 lakh", "five lakh", "bank guarantee release"]
    }
}

# 2. Known information gaps
NOT_VERIFIED = {
    "submission_deadline": "The exact submission deadline has not yet been verified from the extracted evidence.",
    "bid_opening": "The exact bid-opening date and time have not yet been verified.",
    "tender_fee": "A tender fee amount has not yet been verified.",
    "similar_work_threshold": "The exact qualifying similar-work value/threshold has not yet been verified."
}

# 3. Question classification
def classify_tender_question(question):
    q = question.lower().strip()

    # Distinguish the ₹5 lakh bank-guarantee condition from solvency.
    if (
        ("5 lakh" in q or "five lakh" in q)
        and any(x in q for x in ["bank", "guarantee", "deposit"])
    ):
        return "bank_guarantee_release"

    if any(x in q for x in ["solvency", "bank solvency"]):
        return "bank_solvency"

    if any(x in q for x in ["emd", "earnest money"]):
        return "emd"

    if any(x in q for x in ["turnover", "annual revenue"]):
        return "turnover"

    if any(x in q for x in ["estimated cost", "project cost", "tender value", "estimated amount"]):
        return "estimated_cost"

    if any(x in q for x in ["security deposit", "security percentage"]):
        return "security_deposit"

    if any(x in q for x in ["performance guarantee", "performance security"]):
        return "performance_guarantee"

    if any(x in q for x in ["submission deadline", "last date", "bid submission", "closing date"]):
        return "submission_deadline"

    if any(x in q for x in ["opening date", "bid opening", "tender opening"]):
        return "bid_opening"

    if any(x in q for x in ["tender fee", "document fee"]):
        return "tender_fee"

    if any(x in q for x in ["similar work", "experience threshold", "work experience"]):
        return "similar_work_threshold"

    return None


# 4. Evidence formatting for questions not covered by verified facts
def get_rag_evidence(question, top_k=3):
    if "search_tender" not in globals():
        return []

    try:
        results = search_tender(question, top_k=top_k)
    except Exception as e:
        print("Retrieval error:", str(e))
        return []

    evidence = []

    if isinstance(results, pd.DataFrame):
        results = results.to_dict("records")

    if not isinstance(results, (list, tuple)):
        return []

    for item in results:
        if isinstance(item, dict):
            source = item.get("source_file", "Source not identified")
            page = item.get("page_number", "")
            section = item.get("section", "")
            content = item.get("content", item.get("text", ""))
        else:
            continue

        content = str(content).strip()
        if not content:
            continue

        evidence.append({
            "source": str(source),
            "page": str(page),
            "section": str(section),
            "content": content[:900]
        })

    return evidence


# 5. Main answer function
def ask_tendora_verified(question):
    if not isinstance(question, str) or not question.strip():
        return {
            "status": "INVALID QUESTION",
            "answer": "Please enter a tender-related question.",
            "sources": []
        }

    category = classify_tender_question(question)

    # Exact verified answer
    if category in VERIFIED_TENDER_FACTS:
        fact = VERIFIED_TENDER_FACTS[category]
        return {
            "status": "VERIFIED FROM EXTRACTED TENDER EVIDENCE",
            "answer": fact["value"],
            "sources": [fact["source"]]
        }

    # Explicitly disclose known gaps
    if category in NOT_VERIFIED:
        return {
            "status": "NOT VERIFIED",
            "answer": NOT_VERIFIED[category],
            "sources": []
        }

    # Evidence-first fallback for other questions
    evidence = get_rag_evidence(question, top_k=3)

    if not evidence:
        return {
            "status": "INSUFFICIENT EVIDENCE",
            "answer": "I could not retrieve reliable supporting text for this question. Please verify it in the original tender documents.",
            "sources": []
        }

    return {
        "status": "RETRIEVED EVIDENCE — REVIEW REQUIRED",
        "answer": (
            "I found relevant tender passages, but this information has not "
            "been independently verified as a final requirement. Review the "
            "evidence below before making a bidding decision."
        ),
        "sources": [
            f'{x["source"]} | Page/Row: {x["page"]} | {x["section"]}'
            for x in evidence
        ],
        "evidence": [x["content"] for x in evidence]
    }


# 6. Compact display function
def tendora_ask(question):
    result = ask_tendora_verified(question)

    display(Markdown(f"### {result['status']}"))
    display(Markdown(f"**Answer:** {result['answer']}"))

    if result.get("sources"):
        display(Markdown("**Sources:**"))
        for source in result["sources"]:
            display(Markdown(f"- {source}"))

    if result.get("evidence"):
        display(Markdown("**Relevant extracted evidence:**"))
        for i, passage in enumerate(result["evidence"], 1):
            display(Markdown(f"{i}. {passage}"))

    return result


# 7. Short validation tests
test_questions = [
    "What is the EMD amount?",
    "What is the minimum bank solvency?",
    "What is the minimum annual turnover?",
    "What is the estimated project cost?",
    "What is the security deposit percentage?",
    "What is the performance guarantee?",
    "Is the 5 lakh amount the bank solvency requirement?",
    "What is the tender submission deadline?"
]

print("TENDORA VERIFIED Q&A — VALIDATION")
print("=" * 48)

for question in test_questions:
    result = ask_tendora_verified(question)
    print(f"\nQ: {question}")
    print(f"A: {result['answer']}")
    print(f"Status: {result['status']}")
    if result.get("sources"):
        print(f"Source: {result['sources'][0]}")

print("\nCell 23 completed.")


In [ ]:

# ============================================================
# TENDORA | CELL 24
# Bidder Profile + Eligibility Assessment Engine
# ============================================================

import pandas as pd
from IPython.display import display, Markdown

# ------------------------------------------------------------
# 1. TENDER ELIGIBILITY CRITERIA
# ------------------------------------------------------------

TENDER_CRITERIA = {
    "bank_solvency": {
        "label": "Minimum Bank Solvency",
        "minimum": 15.21,
        "unit": "₹ lakh",
        "source": "NIT1TechnicalBid.xls | E-TENDER TECHNICAL BID | Row 30",
        "verified": True
    },
    "turnover": {
        "label": "Financial Turnover",
        "minimum": 38.03,
        "unit": "₹ lakh",
        "period": "Last 3 years",
        "source": "NIT1TechnicalBid.xls | E-TENDER TECHNICAL BID | Row 19",
        "verified": True,
        "interpretation_verified": False
    },
    "similar_work": {
        "label": "Similar Work Experience",
        "period": "Last 7 years",
        "source": "NIT1TechnicalBid.xls | Form C | Row 4",
        "verified": False,
        "note": "Exact qualifying work-value threshold needs verification."
    }
}

# ------------------------------------------------------------
# 2. DEMO BIDDER PROFILE
# Edit these values for another bidder.
# Monetary figures are entered in ₹ lakh.
# ------------------------------------------------------------

bidder_profile = {
    "company_name": "Demo Construction Company",

    "bank_solvency_lakh": 18.00,

    # Enter the qualifying turnover figure in ₹ lakh.
    # Do not enter a guessed average or total.
    # Use None until the correct basis is confirmed.
    "turnover_qualifying_value_lakh": None,

    # Record turnover year-wise for reference.
    "annual_turnover_lakh": {
        "2025-26": None,
        "2024-25": None,
        "2023-24": None
    },

    # Similar-work evidence must be reviewed separately.
    "similar_projects": [],

    # These are demo profile declarations, NOT authenticated documents.
    "documents": {
        "GST Registration": True,
        "PAN Registration": True,
        "EPF Registration": True,
        "ESIC Registration": False,
        "Work Experience Certificates": True,
        "Performance Certificates": True,
        "Bank Solvency Certificate": True,
        "CA Certified Financial Statements": False,
        "Plant and Machinery List": True,
        "Technical Staff List": True,
        "EMD Proof": True,
        "Local Content Undertaking": True,
        "Non Back-to-Back Work Undertaking": False
    }
}

# ------------------------------------------------------------
# 3. ASSESSMENT HELPERS
# ------------------------------------------------------------

def assess_financial_value(value, minimum, label):
    if value is None:
        return {
            "status": "PENDING",
            "details": f"{label} has not been provided."
        }

    try:
        value = float(value)
    except (TypeError, ValueError):
        return {
            "status": "PENDING",
            "details": f"{label} is not a valid numerical value."
        }

    if value < 0:
        return {
            "status": "PENDING",
            "details": f"{label} cannot be negative."
        }

    if value >= minimum:
        return {
            "status": "PASS — CLAIMED",
            "details": f"₹{value:.2f} lakh meets the ₹{minimum:.2f} lakh minimum."
        }

    return {
        "status": "FAIL — CLAIMED",
        "details": f"₹{value:.2f} lakh is below the ₹{minimum:.2f} lakh minimum."
    }


# ------------------------------------------------------------
# 4. MAIN ELIGIBILITY ENGINE
# ------------------------------------------------------------

def assess_bidder(profile):

    results = []

    # Bank solvency
    solvency = assess_financial_value(
        profile.get("bank_solvency_lakh"),
        TENDER_CRITERIA["bank_solvency"]["minimum"],
        "Bank solvency"
    )

    results.append({
        "Requirement": "Bank Solvency",
        "Tender Criterion": "Minimum ₹15.21 lakh",
        "Bidder Information": profile.get("bank_solvency_lakh"),
        "Assessment": solvency["status"],
        "Explanation": solvency["details"],
        "Source": TENDER_CRITERIA["bank_solvency"]["source"]
    })

    # Turnover
    turnover_value = profile.get("turnover_qualifying_value_lakh")

    turnover = assess_financial_value(
        turnover_value,
        TENDER_CRITERIA["turnover"]["minimum"],
        "Qualifying turnover"
    )

    if turnover_value is not None:
        turnover_status = "PENDING — BASIS VERIFICATION"
        turnover_details = (
            turnover["details"]
            + " The tender's average/aggregate interpretation has not been verified."
        )
    else:
        turnover_status = "PENDING"
        turnover_details = (
            "Qualifying turnover figure not provided. "
            "The average/aggregate interpretation also needs verification."
        )

    results.append({
        "Requirement": "Financial Turnover",
        "Tender Criterion": "Minimum ₹38.03 lakh; last 3 years",
        "Bidder Information": turnover_value,
        "Assessment": turnover_status,
        "Explanation": turnover_details,
        "Source": TENDER_CRITERIA["turnover"]["source"]
    })

    # Similar work
    similar_projects = profile.get("similar_projects", [])

    results.append({
        "Requirement": "Similar Work Experience",
        "Tender Criterion": "Relevant work in the last 7 years",
        "Bidder Information": len(similar_projects),
        "Assessment": "PENDING — CRITERION NOT FULLY VERIFIED",
        "Explanation": (
            "The exact qualifying work-value threshold has not been established. "
            "Project documents require manual review."
        ),
        "Source": TENDER_CRITERIA["similar_work"]["source"]
    })

    # Documents
    documents = profile.get("documents", {})

    for document, available in documents.items():

        if available is True:
            status = "PROVIDED — VERIFICATION REQUIRED"
            explanation = "Declared available by bidder; authenticity not checked."

        elif available is False:
            status = "MISSING — AS DECLARED"
            explanation = "Bidder profile declares this document unavailable."

        else:
            status = "PENDING"
            explanation = "Availability has not been specified."

        results.append({
            "Requirement": document,
            "Tender Criterion": "Listed in bidder profile / tender forms",
            "Bidder Information": available,
            "Assessment": status,
            "Explanation": explanation,
            "Source": "NIT1TechnicalBid.xls — relevant technical bid forms; applicability to be checked"
        })

    return pd.DataFrame(results)


# ------------------------------------------------------------
# 5. RUN ASSESSMENT
# ------------------------------------------------------------

eligibility_df = assess_bidder(bidder_profile)

# Count statuses
status_counts = eligibility_df["Assessment"].value_counts()

financial_rows = eligibility_df[
    eligibility_df["Requirement"].isin([
        "Bank Solvency",
        "Financial Turnover",
        "Similar Work Experience"
    ])
]

has_financial_fail = any(
    status.startswith("FAIL")
    for status in financial_rows["Assessment"]
)

has_pending = any(
    status.startswith("PENDING")
    for status in financial_rows["Assessment"]
)

if has_financial_fail:
    overall_status = "POTENTIAL INELIGIBILITY — REVIEW REQUIRED"
elif has_pending:
    overall_status = "INSUFFICIENT INFORMATION"
else:
    overall_status = "FINANCIAL CHECKS PASSED — DOCUMENT VERIFICATION REQUIRED"


# ------------------------------------------------------------
# 6. COMPACT DASHBOARD OUTPUT
# ------------------------------------------------------------

print("=" * 58)
print("TENDORA | BIDDER ELIGIBILITY REPORT")
print("=" * 58)

print(f"\nCompany: {bidder_profile.get('company_name', 'Not provided')}")
print(f"Overall Assessment: {overall_status}")

print("\nASSESSMENT SUMMARY")
print("-" * 35)

for status, count in status_counts.items():
    print(f"{status}: {count}")

print("\nFINANCIAL & EXPERIENCE CHECKS")
print("-" * 35)

display(
    eligibility_df[
        eligibility_df["Requirement"].isin([
            "Bank Solvency",
            "Financial Turnover",
            "Similar Work Experience"
        ])
    ][["Requirement", "Tender Criterion", "Assessment", "Explanation"]]
)

print("\nDOCUMENT CHECKLIST")
print("-" * 35)

display(
    eligibility_df[
        ~eligibility_df["Requirement"].isin([
            "Bank Solvency",
            "Financial Turnover",
            "Similar Work Experience"
        ])
    ][["Requirement", "Assessment", "Explanation"]]
)

# Save detailed report
output_path = "/content/tendora_bidder_eligibility_report.csv"
eligibility_df.to_csv(output_path, index=False)

print(f"\nDetailed report saved: {output_path}")

print("\nIMPORTANT:")
print("- PASS means the bidder's entered figure meets the stated threshold.")
print("- Document availability is self-declared, not authenticated.")
print("- Similar-work qualification is not yet verified.")
print("- This report is an initial screening, not a final eligibility decision.")
print("\nCell 24 completed.")


In [ ]:

# ============================================================
# TENDORA | CELL 25
# Tender Requirement Verification Engine
# ============================================================

import re
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# ------------------------------------------------------------
# 1. CHECK REQUIRED DATA
# ------------------------------------------------------------

if "unified_documents" not in globals():
    raise RuntimeError(
        "unified_documents not found. Please run the document extraction cells first."
    )

if not unified_documents:
    raise RuntimeError("No extracted tender documents are available.")

# ------------------------------------------------------------
# 2. REQUIREMENT DETECTION RULES
# ------------------------------------------------------------

requirement_patterns = {
    "Eligibility Criteria": [
        r"eligible bidder",
        r"eligibility criteria",
        r"pre.?qualification",
        r"qualification criteria",
        r"technical qualification"
    ],
    "Financial Requirements": [
        r"bank solvency",
        r"financial turn.?over",
        r"annual turn.?over",
        r"average annual turnover",
        r"net worth",
        r"financial capacity"
    ],
    "Experience Requirements": [
        r"similar work",
        r"work experience",
        r"completion certificate",
        r"performance certificate",
        r"successfully completed"
    ],
    "Registration and Tax Documents": [
        r"gst registration",
        r"pan card",
        r"permanent account number",
        r"epf registration",
        r"esic registration"
    ],
    "Technical Documents": [
        r"plant and machinery",
        r"technical staff",
        r"technical personnel",
        r"technical proposal",
        r"methodology statement"
    ],
    "Declarations and Undertakings": [
        r"undertaking",
        r"declaration",
        r"affidavit",
        r"non.?blacklisting",
        r"non.?back.?to.?back"
    ],
    "Bid Submission Requirements": [
        r"bid submission",
        r"submit.*document",
        r"upload.*document",
        r"online submission",
        r"technical bid"
    ]
}

mandatory_patterns = [
    r"\bshall submit\b",
    r"\bmust submit\b",
    r"\bmandatory\b",
    r"\bshall be submitted\b",
    r"\bmust be submitted\b",
    r"\brequired to submit\b",
    r"\bcompulsory\b"
]

conditional_patterns = [
    r"\bif applicable\b",
    r"\bwhere applicable\b",
    r"\bif any\b",
    r"\bwherever applicable\b",
    r"\bas applicable\b"
]

supporting_patterns = [
    r"\bshould submit\b",
    r"\bmay submit\b",
    r"\battach.*copy\b",
    r"\benclose.*copy\b"
]

# ------------------------------------------------------------
# 3. CLASSIFICATION HELPERS
# ------------------------------------------------------------

def identify_category(text):
    text_lower = text.lower()

    for category, patterns in requirement_patterns.items():
        if any(re.search(p, text_lower) for p in patterns):
            return category

    return "Other Requirement"


def classify_requirement(text):
    text_lower = text.lower()

    # Conditional wording takes precedence.
    if any(re.search(p, text_lower) for p in conditional_patterns):
        return "CONDITIONAL — VERIFY CONTEXT"

    if any(re.search(p, text_lower) for p in mandatory_patterns):
        return "POTENTIALLY MANDATORY — VERIFY CLAUSE"

    if any(re.search(p, text_lower) for p in supporting_patterns):
        return "SUPPORTING — VERIFY CLAUSE"

    return "NOT ESTABLISHED — REVIEW REQUIRED"


def find_requirement_phrases(text):
    phrases = [
        "shall submit",
        "must submit",
        "mandatory",
        "shall be submitted",
        "must be submitted",
        "required to submit",
        "compulsory",
        "if applicable",
        "where applicable",
        "if any",
        "as applicable",
        "should submit",
        "may submit",
        "attach",
        "enclose"
    ]

    text_lower = text.lower()
    found = [phrase for phrase in phrases if phrase in text_lower]

    return ", ".join(found) if found else "No explicit instruction detected"


# ------------------------------------------------------------
# 4. EXTRACT REQUIREMENT CANDIDATES
# ------------------------------------------------------------

requirement_records = []

for doc in unified_documents:

    source_file = str(doc.get("source_file", "Unknown source"))
    section = str(doc.get("section", ""))
    page_number = doc.get("page_number", "")
    content = str(doc.get("content", ""))

    if not content.strip():
        continue

    # Break large text into manageable sentence-like passages.
    passages = re.split(r"(?<=[.;])\s+|\n+", content)

    for passage in passages:

        passage = re.sub(r"\s+", " ", passage).strip()

        if len(passage) < 25:
            continue

        category = identify_category(passage)

        # Keep only passages matching a known requirement category.
        if category == "Other Requirement":
            continue

        # Exclude irrelevant generic mentions where possible.
        if not any(
            term in passage.lower()
            for term in [
                "submit", "required", "mandatory", "shall",
                "must", "certificate", "registration",
                "undertaking", "experience", "turnover",
                "solvency", "document", "attach", "upload",
                "bidder", "declaration", "affidavit",
                "machinery", "personnel", "technical"
            ]
        ):
            continue

        requirement_records.append({
            "Requirement Category": category,
            "Requirement Text": passage[:1200],
            "Classification": classify_requirement(passage),
            "Instruction Keywords": find_requirement_phrases(passage),
            "Source File": source_file,
            "Page / Row": str(page_number),
            "Section / Sheet": section,
            "Verification Status": "NOT MANUALLY VERIFIED"
        })


# ------------------------------------------------------------
# 5. REMOVE DUPLICATE PASSAGES
# ------------------------------------------------------------

requirement_df = pd.DataFrame(requirement_records)

if not requirement_df.empty:

    requirement_df["dedup_key"] = (
        requirement_df["Source File"].astype(str)
        + requirement_df["Page / Row"].astype(str)
        + requirement_df["Requirement Text"].str.lower().str[:180]
    )

    requirement_df = (
        requirement_df
        .drop_duplicates(subset="dedup_key")
        .drop(columns="dedup_key")
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 6. SAVE REQUIREMENT REGISTER
# ------------------------------------------------------------

csv_path = "/content/tendora_requirement_register.csv"
json_path = "/content/tendora_requirement_register.json"

requirement_df.to_csv(csv_path, index=False)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        requirement_df.to_dict(orient="records"),
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 7. COMPACT SUMMARY
# ------------------------------------------------------------

print("=" * 58)
print("TENDORA | TENDER REQUIREMENT VERIFICATION")
print("=" * 58)

print(f"\nExtracted requirement candidates: {len(requirement_df)}")

if not requirement_df.empty:

    print("\nCATEGORY SUMMARY")
    print("-" * 35)

    category_summary = (
        requirement_df["Requirement Category"]
        .value_counts()
        .rename_axis("Category")
        .reset_index(name="Candidates")
    )

    display(category_summary)

    print("\nCLASSIFICATION SUMMARY")
    print("-" * 35)

    classification_summary = (
        requirement_df["Classification"]
        .value_counts()
        .rename_axis("Classification")
        .reset_index(name="Count")
    )

    display(classification_summary)

    print("\nSAMPLE REQUIREMENT EVIDENCE")
    print("-" * 35)

    display(
        requirement_df[
            [
                "Requirement Category",
                "Requirement Text",
                "Classification",
                "Source File",
                "Page / Row",
                "Section / Sheet"
            ]
        ].head(8)
    )

else:
    print("\nNo requirement candidates detected.")
    print("Check the extracted document content.")

print("\nFILES SAVED")
print(csv_path)
print(json_path)

print("\nIMPORTANT:")
print("- These are extracted candidates, not confirmed mandatory requirements.")
print("- Context and surrounding clauses must be reviewed.")
print("- Conditional requirements must not be treated as universally mandatory.")
print("- Original tender documents remain the authority.")

print("\nCell 25 completed.")


In [ ]:

# ============================================================
# TENDORA | CELL 26
# Evidence-Based Requirement Verification
# ============================================================

import pandas as pd
import re
import json
from IPython.display import display, Markdown

# ------------------------------------------------------------
# 1. VALIDATE CELL 25 OUTPUT
# ------------------------------------------------------------

if "requirement_df" not in globals():
    raise RuntimeError("Run Cell 25 first. requirement_df is missing.")

if requirement_df.empty:
    raise RuntimeError("No requirement candidates found in Cell 25.")

df = requirement_df.copy()

required_columns = [
    "Requirement Category",
    "Requirement Text",
    "Classification",
    "Source File",
    "Page / Row",
    "Section / Sheet"
]

missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")


# ------------------------------------------------------------
# 2. NORMALIZE REQUIREMENT TEXT
# ------------------------------------------------------------

def normalize_requirement(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9₹. ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["Normalized Text"] = df["Requirement Text"].apply(normalize_requirement)


# ------------------------------------------------------------
# 3. GROUP RELATED REQUIREMENTS
# ------------------------------------------------------------

def identify_requirement_topic(text):

    t = text.lower()

    topics = {
        "Bank Solvency": ["bank solvency", "solvency certificate"],
        "Financial Turnover": ["turnover", "financial turn"],
        "Similar Work Experience": ["similar work", "work experience"],
        "GST Registration": ["gst registration", "gst certificate"],
        "PAN Registration": ["pan card", "pan registration"],
        "EPF Registration": ["epf registration", "provident fund"],
        "ESIC Registration": ["esic registration", "employee state insurance"],
        "Financial Statements": ["profit and loss", "balance sheet", "financial statement"],
        "Plant and Machinery": ["plant and machinery", "machinery list"],
        "Technical Staff": ["technical staff", "technical personnel"],
        "EMD": ["earnest money", "emd"],
        "Undertakings": ["undertaking", "declaration", "affidavit"],
        "Bid Submission": ["bid submission", "upload", "submit the bid"]
    }

    for topic, keywords in topics.items():
        if any(keyword in t for keyword in keywords):
            return topic

    return "Other / General Requirement"


df["Requirement Topic"] = df["Requirement Text"].apply(
    identify_requirement_topic
)


# ------------------------------------------------------------
# 4. ASSIGN REVIEW PRIORITY
# ------------------------------------------------------------

def assign_priority(row):

    topic = row["Requirement Topic"]
    classification = row["Classification"]

    high_priority_topics = [
        "Bank Solvency",
        "Financial Turnover",
        "Similar Work Experience",
        "EMD",
        "Bid Submission"
    ]

    if topic in high_priority_topics:
        return "HIGH"

    if "POTENTIALLY MANDATORY" in classification:
        return "HIGH"

    if "CONDITIONAL" in classification:
        return "MEDIUM"

    return "REVIEW"


df["Review Priority"] = df.apply(assign_priority, axis=1)


# ------------------------------------------------------------
# 5. CREATE EVIDENCE-BASED REQUIREMENT REGISTER
# ------------------------------------------------------------

df["Verification Status"] = "PENDING MANUAL CLAUSE REVIEW"

df["Final Requirement Type"] = "NOT CONFIRMED"

df["Reviewer Notes"] = (
    "Check the original tender clause and surrounding context "
    "before confirming applicability."
)

# Keep all original source references.
register_columns = [
    "Requirement Topic",
    "Requirement Category",
    "Requirement Text",
    "Classification",
    "Review Priority",
    "Source File",
    "Page / Row",
    "Section / Sheet",
    "Verification Status",
    "Final Requirement Type",
    "Reviewer Notes"
]

requirement_register = df[register_columns].copy()


# ------------------------------------------------------------
# 6. CREATE GROUPED SUMMARY
# ------------------------------------------------------------

grouped_summary = (
    requirement_register
    .groupby("Requirement Topic")
    .agg(
        Candidate_Count=("Requirement Text", "count"),
        Source_Count=("Source File", "nunique"),
        High_Priority_Count=(
            "Review Priority",
            lambda x: (x == "HIGH").sum()
        )
    )
    .reset_index()
    .sort_values(
        by=["High_Priority_Count", "Candidate_Count"],
        ascending=False
    )
)


# ------------------------------------------------------------
# 7. SAVE OUTPUTS
# ------------------------------------------------------------

csv_path = "/content/tendora_verified_requirement_register.csv"
json_path = "/content/tendora_verified_requirement_register.json"
summary_path = "/content/tendora_requirement_summary.csv"

requirement_register.to_csv(csv_path, index=False)
grouped_summary.to_csv(summary_path, index=False)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        requirement_register.to_dict(orient="records"),
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 8. DISPLAY COMPACT RESULTS
# ------------------------------------------------------------

print("=" * 58)
print("TENDORA | REQUIREMENT REVIEW REGISTER")
print("=" * 58)

print(f"\nTotal requirement candidates: {len(requirement_register)}")
print(
    "Unique requirement topics:",
    requirement_register["Requirement Topic"].nunique()
)

print("\nGROUPED REQUIREMENT SUMMARY")
print("-" * 35)
display(grouped_summary.head(12))

print("\nHIGH-PRIORITY REQUIREMENTS")
print("-" * 35)

high_priority = requirement_register[
    requirement_register["Review Priority"] == "HIGH"
]

display(
    high_priority[
        [
            "Requirement Topic",
            "Requirement Text",
            "Source File",
            "Page / Row",
            "Verification Status"
        ]
    ].head(8)
)

print("\nSAVED FILES")
print(csv_path)
print(json_path)
print(summary_path)

print("\nIMPORTANT:")
print("- Grouping is based on keyword matching, not legal interpretation.")
print("- Repeated clauses remain traceable to their individual sources.")
print("- No requirement is automatically confirmed as mandatory.")
print("- Manual verification against the original tender remains necessary.")

print("\nCell 26 completed.")


In [ ]:

# ============================================================
# TENDORA | CELL 27
# Bidder Document Upload & Evidence Matching
# ============================================================

import os
import re
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# ------------------------------------------------------------
# 1. CHECK DEPENDENCIES AND PREVIOUS CELLS
# ------------------------------------------------------------

try:
    import fitz
    import pytesseract
    from PIL import Image
except ImportError as e:
    raise RuntimeError(
        f"Missing dependency: {e}. Install the missing package and rerun."
    )

if "requirement_register" not in globals():
    raise RuntimeError("Run Cell 26 first.")

# ------------------------------------------------------------
# 2. DOCUMENT TYPE DETECTION
# ------------------------------------------------------------

document_patterns = {
    "GST Registration": [
        r"\bgst\b", r"goods and services tax", r"gstin"
    ],
    "PAN Registration": [
        r"\bpan\b", r"permanent account number"
    ],
    "EPF Registration": [
        r"\bepf\b", r"provident fund", r"epfo"
    ],
    "ESIC Registration": [
        r"\besic\b", r"employee state insurance"
    ],
    "Bank Solvency Certificate": [
        r"bank solvency", r"solvency certificate"
    ],
    "Financial Statements": [
        r"profit and loss", r"balance sheet",
        r"audited accounts", r"financial statements"
    ],
    "Turnover Certificate": [
        r"annual turnover", r"gross turnover",
        r"financial turnover", r"turnover certificate"
    ],
    "Work Experience Certificate": [
        r"work experience", r"completion certificate",
        r"similar work", r"work order"
    ],
    "Performance Certificate": [
        r"performance certificate", r"satisfactory performance",
        r"performance report"
    ],
    "Plant and Machinery List": [
        r"plant and machinery", r"machinery list",
        r"equipment list"
    ],
    "Technical Staff List": [
        r"technical staff", r"technical personnel",
        r"staff details"
    ],
    "EMD Proof": [
        r"earnest money", r"\bemd\b",
        r"bid security", r"payment receipt"
    ],
    "Local Content Undertaking": [
        r"local content", r"class.i local supplier",
        r"class.ii local supplier"
    ],
    "Non Back-to-Back Work Undertaking": [
        r"non.back.to.back", r"back.to.back undertaking"
    ]
}


def detect_document_type(filename, extracted_text):

    combined = (filename + " " + extracted_text[:12000]).lower()

    matches = []

    for document_type, patterns in document_patterns.items():
        score = sum(
            1 for pattern in patterns
            if re.search(pattern, combined, re.IGNORECASE)
        )

        if score > 0:
            matches.append((document_type, score))

    matches.sort(key=lambda x: x[1], reverse=True)

    return matches


# ------------------------------------------------------------
# 3. TEXT EXTRACTION
# ------------------------------------------------------------

def extract_pdf_text(path):

    pages = []

    with fitz.open(path) as pdf:

        for page_number, page in enumerate(pdf, start=1):

            text = page.get_text("text").strip()
            method = "PDF TEXT"

            # OCR fallback for scanned pages.
            if len(text) < 40:
                try:
                    pix = page.get_pixmap(
                        matrix=fitz.Matrix(2, 2),
                        alpha=False
                    )

                    image_path = f"/tmp/tendora_ocr_{page_number}.png"
                    pix.save(image_path)

                    image = Image.open(image_path)
                    text = pytesseract.image_to_string(image)
                    method = "OCR"

                except Exception:
                    method = "OCR FAILED"

            pages.append({
                "page": page_number,
                "text": text.strip(),
                "method": method
            })

    return pages


def extract_image_text(path):

    image = Image.open(path)
    text = pytesseract.image_to_string(image)

    return [{
        "page": 1,
        "text": text.strip(),
        "method": "IMAGE OCR"
    }]


def extract_text_file(path):

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    return [{
        "page": 1,
        "text": text.strip(),
        "method": "TEXT EXTRACTION"
    }]


# ------------------------------------------------------------
# 4. UPLOAD BIDDER DOCUMENTS
# ------------------------------------------------------------

print("=" * 58)
print("TENDORA | BIDDER DOCUMENT UPLOAD")
print("=" * 58)

print("\nUpload actual bidder supporting documents.")
print("You may select multiple files.")
print("Supported: PDF, JPG, JPEG, PNG, TXT")

try:
    from google.colab import files

    uploaded_files = files.upload()

except ImportError:
    raise RuntimeError(
        "This upload cell is designed for Google Colab."
    )


# ------------------------------------------------------------
# 5. PROCESS UPLOADED DOCUMENTS
# ------------------------------------------------------------

upload_folder = Path("/content/tendora_bidder_uploads")
upload_folder.mkdir(exist_ok=True)

document_results = []
page_evidence = []

for filename, file_bytes in uploaded_files.items():

    extension = Path(filename).suffix.lower()
    safe_name = Path(filename).name
    save_path = upload_folder / safe_name

    with open(save_path, "wb") as f:
        f.write(file_bytes)

    try:

        if extension == ".pdf":
            pages = extract_pdf_text(str(save_path))

        elif extension in [".jpg", ".jpeg", ".png"]:
            pages = extract_image_text(str(save_path))

        elif extension == ".txt":
            pages = extract_text_file(str(save_path))

        else:
            document_results.append({
                "Uploaded File": filename,
                "Detected Document Type": "Unsupported",
                "Status": "NOT PROCESSED",
                "Pages": 0,
                "Extracted Characters": 0
            })
            continue

        combined_text = "\n".join(
            page["text"] for page in pages
        )

        matches = detect_document_type(filename, combined_text)

        detected_type = (
            matches[0][0] if matches else "UNIDENTIFIED DOCUMENT"
        )

        if detected_type == "UNIDENTIFIED DOCUMENT":
            status = "MANUAL IDENTIFICATION REQUIRED"
        else:
            status = "POTENTIAL DOCUMENT MATCH — REVIEW REQUIRED"

        document_results.append({
            "Uploaded File": filename,
            "Detected Document Type": detected_type,
            "Status": status,
            "Pages": len(pages),
            "Extracted Characters": len(combined_text)
        })

        for page in pages:

            if page["text"].strip():

                page_evidence.append({
                    "Uploaded File": filename,
                    "Detected Document Type": detected_type,
                    "Page": page["page"],
                    "Extraction Method": page["method"],
                    "Extracted Text": page["text"][:5000]
                })

    except Exception as e:

        document_results.append({
            "Uploaded File": filename,
            "Detected Document Type": "Extraction Error",
            "Status": str(e)[:200],
            "Pages": 0,
            "Extracted Characters": 0
        })


# ------------------------------------------------------------
# 6. MATCH UPLOADS AGAINST REQUIREMENT REGISTER
# ------------------------------------------------------------

matching_results = []

for uploaded in document_results:

    detected = uploaded["Detected Document Type"]

    if detected in ["Unsupported", "UNIDENTIFIED DOCUMENT", "Extraction Error"]:
        matching_results.append({
            "Uploaded File": uploaded["Uploaded File"],
            "Detected Type": detected,
            "Matching Requirement": "Not established",
            "Matching Status": "MANUAL REVIEW REQUIRED"
        })
        continue

    matching_requirement = detected

    matching_results.append({
        "Uploaded File": uploaded["Uploaded File"],
        "Detected Type": detected,
        "Matching Requirement": matching_requirement,
        "Matching Status": "POTENTIAL MATCH — REVIEW REQUIRED"
    })


# ------------------------------------------------------------
# 7. REQUIREMENT COVERAGE OVERVIEW
# ------------------------------------------------------------

uploaded_types = {
    x["Detected Document Type"]
    for x in document_results
    if x["Detected Document Type"] not in [
        "Unsupported",
        "UNIDENTIFIED DOCUMENT",
        "Extraction Error"
    ]
}

coverage_rows = []

for document_type in document_patterns:

    if document_type in uploaded_types:
        status = "POTENTIAL UPLOAD MATCH"
    else:
        status = "NO MATCHING UPLOAD DETECTED"

    coverage_rows.append({
        "Document Type": document_type,
        "Upload Status": status
    })

coverage_df = pd.DataFrame(coverage_rows)


# ------------------------------------------------------------
# 8. SAVE OUTPUTS
# ------------------------------------------------------------

documents_df = pd.DataFrame(document_results)
matching_df = pd.DataFrame(matching_results)
evidence_df = pd.DataFrame(page_evidence)

documents_df.to_csv(
    "/content/tendora_uploaded_documents.csv",
    index=False
)

matching_df.to_csv(
    "/content/tendora_document_matching.csv",
    index=False
)

coverage_df.to_csv(
    "/content/tendora_document_coverage.csv",
    index=False
)

evidence_df.to_json(
    "/content/tendora_uploaded_document_evidence.json",
    orient="records",
    indent=2,
    force_ascii=False
)


# ------------------------------------------------------------
# 9. COMPACT OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 58)
print("TENDORA | DOCUMENT PROCESSING REPORT")
print("=" * 58)

print(f"\nFiles uploaded: {len(uploaded_files)}")
print(f"Files processed: {len(page_evidence)} pages with extracted text")

print("\nDOCUMENT DETECTION")
print("-" * 35)

if not documents_df.empty:
    display(documents_df)

print("\nREQUIREMENT MATCHING")
print("-" * 35)

if not matching_df.empty:
    display(matching_df)

print("\nDOCUMENT COVERAGE")
print("-" * 35)

display(
    coverage_df[
        coverage_df["Upload Status"] == "POTENTIAL UPLOAD MATCH"
    ]
)

print("\nOUTPUT FILES SAVED:")
print("/content/tendora_uploaded_documents.csv")
print("/content/tendora_document_matching.csv")
print("/content/tendora_document_coverage.csv")
print("/content/tendora_uploaded_document_evidence.json")

print("\nIMPORTANT:")
print("- A detected document type is only a potential match.")
print("- This cell does not authenticate certificates or signatures.")
print("- It does not establish whether a document is legally mandatory.")
print("- Human review is required before marking a requirement satisfied.")

print("\nCell 27 completed.")


In [ ]:

# ============================================================
# TENDORA | CELL 27
# Smart Document Classification & Bidder Evidence Matching
# ============================================================

import os
import re
import json
import pandas as pd
import fitz
import pytesseract

from pathlib import Path
from PIL import Image
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

UPLOAD_FOLDER = Path("/content/tendora_bidder_uploads")
UPLOAD_FOLDER.mkdir(exist_ok=True)

# Reuse existing uploads if available.
existing_files = list(UPLOAD_FOLDER.iterdir())

if not existing_files:
    print("Upload bidder supporting documents.")

    from google.colab import files
    uploaded_files = files.upload()

    for filename, file_bytes in uploaded_files.items():
        with open(UPLOAD_FOLDER / Path(filename).name, "wb") as f:
            f.write(file_bytes)

# ------------------------------------------------------------
# 2. DOCUMENT CLASSIFICATION RULES
# ------------------------------------------------------------

tender_filename_patterns = [
    r"schedule",
    r"tender",
    r"technicalspecification",
    r"technical_specification",
    r"nit",
    r"boq",
    r"noticeinviting",
    r"bid_document",
    r"tender_document"
]

tender_content_patterns = [
    r"notice inviting tender",
    r"government of india",
    r"estimated cost",
    r"earnest money",
    r"tender documents",
    r"schedule of quantities",
    r"terms and conditions",
    r"contract agreement",
    r"bhabha atomic research centre"
]

bidder_document_patterns = {
    "GST Registration": [
        r"\bgstin\b", r"goods and services tax",
        r"registration certificate"
    ],
    "PAN Registration": [
        r"permanent account number", r"\bpan\b"
    ],
    "EPF Registration": [
        r"employees provident fund", r"\bepfo\b", r"\bpf code\b"
    ],
    "ESIC Registration": [
        r"employee state insurance", r"\besic\b"
    ],
    "Bank Solvency Certificate": [
        r"bank solvency", r"solvency certificate"
    ],
    "Financial Statements": [
        r"profit and loss account", r"balance sheet",
        r"audited financial statements"
    ],
    "Turnover Certificate": [
        r"annual turnover", r"gross turnover",
        r"turnover certificate"
    ],
    "Work Experience Certificate": [
        r"work completion certificate", r"work experience",
        r"similar work", r"work order"
    ],
    "Performance Certificate": [
        r"performance certificate", r"satisfactory performance"
    ],
    "Plant and Machinery List": [
        r"plant and machinery", r"machinery list"
    ],
    "Technical Staff List": [
        r"technical staff", r"technical personnel"
    ],
    "EMD Proof": [
        r"earnest money deposit", r"\bemd\b", r"bid security"
    ],
    "Local Content Undertaking": [
        r"local content", r"class i local supplier",
        r"class ii local supplier"
    ],
    "Non Back-to-Back Work Undertaking": [
        r"non.?back.?to.?back"
    ]
}

# ------------------------------------------------------------
# 3. TEXT EXTRACTION
# ------------------------------------------------------------

def extract_document_pages(path):

    extension = Path(path).suffix.lower()
    pages = []

    if extension == ".pdf":

        with fitz.open(path) as pdf:

            for page_number, page in enumerate(pdf, 1):

                text = page.get_text("text").strip()
                method = "PDF TEXT"

                if len(text) < 40:
                    try:
                        pix = page.get_pixmap(
                            matrix=fitz.Matrix(2, 2),
                            alpha=False
                        )
                        image = Image.frombytes(
                            "RGB",
                            [pix.width, pix.height],
                            pix.samples
                        )
                        text = pytesseract.image_to_string(image)
                        method = "OCR"
                    except Exception:
                        method = "OCR FAILED"

                pages.append({
                    "page": page_number,
                    "text": text.strip(),
                    "method": method
                })

    elif extension in [".jpg", ".jpeg", ".png"]:

        image = Image.open(path)
        text = pytesseract.image_to_string(image)

        pages.append({
            "page": 1,
            "text": text.strip(),
            "method": "IMAGE OCR"
        })

    elif extension == ".txt":

        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

        pages.append({
            "page": 1,
            "text": text.strip(),
            "method": "TEXT EXTRACTION"
        })

    return pages


# ------------------------------------------------------------
# 4. DISTINGUISH TENDER AND BIDDER DOCUMENTS
# ------------------------------------------------------------

def classify_document(filename, text):

    filename_lower = filename.lower()
    text_lower = text.lower()

    filename_match = any(
        re.search(pattern, filename_lower)
        for pattern in tender_filename_patterns
    )

    tender_content_score = sum(
        1 for pattern in tender_content_patterns
        if re.search(pattern, text_lower)
    )

    # Avoid classifying a tender schedule as a bidder certificate.
    if filename_match or tender_content_score >= 3:
        return "TENDER DOCUMENT", "Not applicable to bidder upload matching"

    bidder_matches = []

    for document_type, patterns in bidder_document_patterns.items():

        score = sum(
            1 for pattern in patterns
            if re.search(pattern, text_lower)
        )

        if score > 0:
            bidder_matches.append((document_type, score))

    bidder_matches.sort(key=lambda x: x[1], reverse=True)

    if bidder_matches:
        return "BIDDER DOCUMENT", bidder_matches[0][0]

    return "UNKNOWN DOCUMENT", "Manual identification required"


# ------------------------------------------------------------
# 5. PROCESS DOCUMENTS
# ------------------------------------------------------------

document_results = []
page_evidence = []

for path in sorted(UPLOAD_FOLDER.iterdir()):

    if not path.is_file():
        continue

    filename = path.name
    extension = path.suffix.lower()

    if extension not in [".pdf", ".jpg", ".jpeg", ".png", ".txt"]:
        continue

    try:

        pages = extract_document_pages(str(path))
        combined_text = "\n".join(p["text"] for p in pages)

        document_class, detected_type = classify_document(
            filename,
            combined_text
        )

        if document_class == "TENDER DOCUMENT":
            status = "TENDER REFERENCE — NOT A BIDDER CERTIFICATE"

        elif document_class == "BIDDER DOCUMENT":
            status = "POTENTIAL MATCH — MANUAL REVIEW REQUIRED"

        else:
            status = "MANUAL IDENTIFICATION REQUIRED"

        document_results.append({
            "Uploaded File": filename,
            "Document Class": document_class,
            "Detected Document Type": detected_type,
            "Status": status,
            "Pages": len(pages),
            "Extracted Characters": len(combined_text)
        })

        for page in pages:

            if page["text"].strip():
                page_evidence.append({
                    "Uploaded File": filename,
                    "Document Class": document_class,
                    "Detected Document Type": detected_type,
                    "Page": page["page"],
                    "Extraction Method": page["method"],
                    "Extracted Text": page["text"][:5000]
                })

    except Exception as e:

        document_results.append({
            "Uploaded File": filename,
            "Document Class": "PROCESSING ERROR",
            "Detected Document Type": "Not identified",
            "Status": str(e)[:200],
            "Pages": 0,
            "Extracted Characters": 0
        })


# ------------------------------------------------------------
# 6. MATCH ONLY BIDDER DOCUMENTS
# ------------------------------------------------------------

documents_df = pd.DataFrame(document_results)
evidence_df = pd.DataFrame(page_evidence)

bidder_matches = documents_df[
    documents_df["Document Class"] == "BIDDER DOCUMENT"
].copy()

if not bidder_matches.empty:

    matching_df = bidder_matches[
        [
            "Uploaded File",
            "Detected Document Type",
            "Status"
        ]
    ].rename(columns={
        "Detected Document Type": "Matching Requirement",
        "Status": "Matching Status"
    })

else:

    matching_df = pd.DataFrame(columns=[
        "Uploaded File",
        "Matching Requirement",
        "Matching Status"
    ])


# ------------------------------------------------------------
# 7. DOCUMENT COVERAGE
# ------------------------------------------------------------

uploaded_types = set(
    bidder_matches["Detected Document Type"].tolist()
)

coverage_rows = []

for document_type in bidder_document_patterns:

    coverage_rows.append({
        "Document Type": document_type,
        "Upload Status": (
            "POTENTIAL UPLOAD MATCH"
            if document_type in uploaded_types
            else "NO MATCHING UPLOAD DETECTED"
        )
    })

coverage_df = pd.DataFrame(coverage_rows)


# ------------------------------------------------------------
# 8. SAVE RESULTS
# ------------------------------------------------------------

documents_df.to_csv(
    "/content/tendora_uploaded_documents.csv",
    index=False
)

matching_df.to_csv(
    "/content/tendora_document_matching.csv",
    index=False
)

coverage_df.to_csv(
    "/content/tendora_document_coverage.csv",
    index=False
)

evidence_df.to_json(
    "/content/tendora_uploaded_document_evidence.json",
    orient="records",
    indent=2,
    force_ascii=False
)


# ------------------------------------------------------------
# 9. COMPACT REPORT
# ------------------------------------------------------------

print("=" * 58)
print("TENDORA | SMART DOCUMENT CLASSIFICATION")
print("=" * 58)

print(f"\nTotal uploaded files: {len(documents_df)}")

if not documents_df.empty:

    print("\nDOCUMENT CLASSIFICATION")
    display(
        documents_df[
            [
                "Uploaded File",
                "Document Class",
                "Detected Document Type",
                "Status"
            ]
        ]
    )

print("\nBIDDER DOCUMENT MATCHES")
if not matching_df.empty:
    display(matching_df)
else:
    print("No bidder certificates identified yet.")

print("\nDOCUMENT COVERAGE")
display(
    coverage_df[
        coverage_df["Upload Status"] == "POTENTIAL UPLOAD MATCH"
    ]
)

print("\nSAVED OUTPUTS:")
print("/content/tendora_uploaded_documents.csv")
print("/content/tendora_document_matching.csv")
print("/content/tendora_document_coverage.csv")
print("/content/tendora_uploaded_document_evidence.json")

print("\nCell 27 completed.")


In [ ]:

# ============================================================
# TENDORA | CELL 28
# Synthetic Bidder Document Generator
# Dependency-safe version
# ============================================================

import sys
import subprocess
import importlib.util

# ------------------------------------------------------------
# 1. INSTALL MISSING DEPENDENCY
# ------------------------------------------------------------

if importlib.util.find_spec("reportlab") is None:
    print("Installing ReportLab...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "reportlab"
    ])

print("ReportLab ready.")

# ------------------------------------------------------------
# 2. IMPORTS
# ------------------------------------------------------------

import zipfile
from pathlib import Path

from reportlab.lib.pagesizes import A4
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
)
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm

# ------------------------------------------------------------
# 3. OUTPUT FOLDER AND STYLES
# ------------------------------------------------------------

SAMPLE_DIR = Path("/content/tendora_synthetic_documents")
SAMPLE_DIR.mkdir(exist_ok=True)

styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    "DocumentTitle",
    parent=styles["Title"],
    fontSize=17,
    leading=22,
    alignment=1,
    spaceAfter=15
)

normal_style = ParagraphStyle(
    "DocumentBody",
    parent=styles["BodyText"],
    fontSize=10,
    leading=15,
    spaceAfter=7
)

warning_style = ParagraphStyle(
    "SyntheticWarning",
    parent=styles["Heading2"],
    textColor=colors.red,
    alignment=1,
    fontSize=12
)

# ------------------------------------------------------------
# 4. PDF GENERATOR
# ------------------------------------------------------------

def create_sample_pdf(filename, title, subtitle, details):

    path = SAMPLE_DIR / filename

    doc = SimpleDocTemplate(
        str(path),
        pagesize=A4,
        rightMargin=22*mm,
        leftMargin=22*mm,
        topMargin=22*mm,
        bottomMargin=22*mm
    )

    story = [
        Paragraph(
            "SYNTHETIC – NOT VALID FOR OFFICIAL USE",
            warning_style
        ),
        Spacer(1, 10),
        Paragraph(title, title_style),
        Paragraph(subtitle, normal_style),
        Spacer(1, 15)
    ]

    table_data = [["Field", "Sample Information"]]

    for key, value in details.items():
        table_data.append([key, value])

    table = Table(table_data, colWidths=[55*mm, 105*mm])

    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#DCEFE7")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.HexColor("#145A40")),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTNAME", (0, 1), (0, -1), "Helvetica-Bold"),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("FONTSIZE", (0, 0), (-1, -1), 9),
        ("LEFTPADDING", (0, 0), (-1, -1), 8),
        ("RIGHTPADDING", (0, 0), (-1, -1), 8),
        ("TOPPADDING", (0, 0), (-1, -1), 8),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 8)
    ]))

    story.extend([
        table,
        Spacer(1, 20),
        Paragraph(
            "Fictional document generated exclusively for "
            "TENDORA software testing. This is not an official "
            "certificate, registration, financial statement, "
            "or proof of eligibility.",
            normal_style
        )
    ])

    def watermark(canvas, document):
        canvas.saveState()
        canvas.setFillColor(colors.Color(0.85, 0.1, 0.1, alpha=0.15))
        canvas.setFont("Helvetica-Bold", 27)
        canvas.translate(105*mm, 148*mm)
        canvas.rotate(35)
        canvas.drawCentredString(0, 0, "SYNTHETIC TEST DATA")
        canvas.restoreState()

    doc.build(story, onFirstPage=watermark, onLaterPages=watermark)

    return path


# ------------------------------------------------------------
# 5. SYNTHETIC SAMPLE DATA
# ------------------------------------------------------------

sample_documents = [
    {
        "filename": "01_GST_Registration_SAMPLE.pdf",
        "title": "GST REGISTRATION – SAMPLE",
        "subtitle": "Fictional business registration information",
        "details": {
            "Company Name": "Demo Infrastructure Private Limited",
            "GSTIN": "27ABCDE1234F1Z5",
            "Registration Type": "Regular",
            "State": "Maharashtra",
            "Registration Date": "01/04/2025"
        }
    },
    {
        "filename": "02_PAN_Registration_SAMPLE.pdf",
        "title": "PAN INFORMATION – SAMPLE",
        "subtitle": "Fictional tax identification information",
        "details": {
            "Company Name": "Demo Infrastructure Private Limited",
            "PAN": "ABCDE1234F",
            "Entity Type": "Company",
            "Issue Date": "15/03/2025"
        }
    },
    {
        "filename": "03_Bank_Solvency_SAMPLE.pdf",
        "title": "BANK SOLVENCY CERTIFICATE – SAMPLE",
        "subtitle": "Fictional financial capacity information",
        "details": {
            "Company Name": "Demo Infrastructure Private Limited",
            "Certificate Number": "SYNTH-SOL-2026-001",
            "Sample Bank": "Example Cooperative Bank",
            "Solvency Amount": "INR 18,00,000",
            "Issue Date": "01/10/2026",
            "Validity": "01/10/2027"
        }
    },
    {
        "filename": "04_Turnover_Statement_SAMPLE.pdf",
        "title": "CA-CERTIFIED TURNOVER – SAMPLE",
        "subtitle": "Fictional financial turnover information",
        "details": {
            "Company Name": "Demo Infrastructure Private Limited",
            "Certificate Number": "SYNTH-TURN-2026-001",
            "FY 2023-24": "INR 14,00,000",
            "FY 2024-25": "INR 16,00,000",
            "FY 2025-26": "INR 18,00,000",
            "Total Sample Turnover": "INR 48,00,000"
        }
    },
    {
        "filename": "05_Work_Experience_SAMPLE.pdf",
        "title": "WORK EXPERIENCE CERTIFICATE – SAMPLE",
        "subtitle": "Fictional project experience information",
        "details": {
            "Company Name": "Demo Infrastructure Private Limited",
            "Certificate Number": "SYNTH-EXP-2026-001",
            "Project": "Sample Industrial Piping Work",
            "Client": "Example Engineering Authority",
            "Completion Date": "30/06/2025",
            "Sample Project Value": "INR 25,00,000"
        }
    },
    {
        "filename": "06_Performance_Certificate_SAMPLE.pdf",
        "title": "PERFORMANCE CERTIFICATE – SAMPLE",
        "subtitle": "Fictional project performance information",
        "details": {
            "Company Name": "Demo Infrastructure Private Limited",
            "Certificate Number": "SYNTH-PERF-2026-001",
            "Project": "Sample Industrial Piping Work",
            "Performance": "Satisfactory – Sample Record",
            "Issue Date": "15/07/2025"
        }
    }
]

# ------------------------------------------------------------
# 6. GENERATE DOCUMENTS
# ------------------------------------------------------------

generated_files = []

for item in sample_documents:

    path = create_sample_pdf(
        item["filename"],
        item["title"],
        item["subtitle"],
        item["details"]
    )

    generated_files.append(path)

# ------------------------------------------------------------
# 7. CREATE ZIP
# ------------------------------------------------------------

zip_path = "/content/TENDORA_Synthetic_Bidder_Documents.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:

    for path in generated_files:
        archive.write(path, arcname=path.name)

# ------------------------------------------------------------
# 8. VALIDATE AND DISPLAY
# ------------------------------------------------------------

assert len(generated_files) == 6
assert Path(zip_path).exists()

print("=" * 55)
print("TENDORA | SYNTHETIC DOCUMENT GENERATOR")
print("=" * 55)

print(f"\nDocuments generated: {len(generated_files)}")

for path in generated_files:
    print("✓", path.name)

print("\nZIP package created successfully:")
print(zip_path)

print("\nCell 28 completed.")

# Download ZIP
from google.colab import files
files.download(zip_path)


In [ ]:

# ============================================================
# TENDORA | CELL 29
# Automated Certificate Information Extraction
# ============================================================

import re
import json
import pandas as pd
import fitz

from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 1. VALIDATE INPUT FILES
# ------------------------------------------------------------

SAMPLE_DIR = Path("/content/tendora_synthetic_documents")

if not SAMPLE_DIR.exists():
    raise RuntimeError("Synthetic document folder not found. Run Cell 28 first.")

pdf_files = sorted(SAMPLE_DIR.glob("*.pdf"))

if not pdf_files:
    raise RuntimeError("No synthetic PDF documents found.")

print("=" * 58)
print("TENDORA | AUTOMATED CERTIFICATE EXTRACTION")
print("=" * 58)

print(f"\nDocuments detected: {len(pdf_files)}")


# ------------------------------------------------------------
# 2. PDF TEXT EXTRACTION
# ------------------------------------------------------------

def extract_pdf_text(path):

    page_texts = []

    with fitz.open(path) as pdf:

        for page_number, page in enumerate(pdf, start=1):

            text = page.get_text("text")

            page_texts.append({
                "page": page_number,
                "text": text
            })

    return "\n".join(x["text"] for x in page_texts)


# ------------------------------------------------------------
# 3. FIELD EXTRACTION HELPERS
# ------------------------------------------------------------

def extract_field(text, labels):

    for label in labels:

        pattern = (
            rf"{re.escape(label)}\s*[:\-]\s*"
            rf"([^\n]+)"
        )

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:
            return match.group(1).strip()

    return None


def extract_company_name(text):

    return extract_field(
        text,
        ["Company Name", "Business Name", "Name of Company"]
    )


def extract_certificate_number(text):

    return extract_field(
        text,
        ["Certificate Number", "Certificate No", "Registration Number"]
    )


def extract_dates(text):

    date_pattern = (
        r"\b\d{1,2}[/-]\d{1,2}[/-]\d{4}\b"
        r"|\b\d{4}-\d{2}-\d{2}\b"
    )

    return re.findall(date_pattern, text)


def extract_amounts(text):

    amount_pattern = (
        r"(?:INR|Rs\.?|₹)\s*"
        r"[\d,]+(?:\.\d{1,2})?"
    )

    return re.findall(amount_pattern, text, flags=re.IGNORECASE)


def extract_identifiers(text):

    identifiers = {}

    patterns = {
        "GSTIN": r"\b\d{2}[A-Z]{5}\d{4}[A-Z][A-Z0-9]Z[A-Z0-9]\b",
        "PAN": r"\b[A-Z]{5}\d{4}[A-Z]\b"
    }

    for name, pattern in patterns.items():

        match = re.search(pattern, text, re.IGNORECASE)

        identifiers[name] = (
            match.group(0) if match else None
        )

    return identifiers


# ------------------------------------------------------------
# 4. DOCUMENT TYPE DETECTION
# ------------------------------------------------------------

def identify_type(filename, text):

    combined = (filename + " " + text).lower()

    patterns = {
        "GST Registration": ["gst", "gstin"],
        "PAN Registration": ["pan"],
        "Bank Solvency Certificate": ["bank solvency"],
        "Turnover Certificate": ["turnover"],
        "Work Experience Certificate": ["work experience"],
        "Performance Certificate": ["performance certificate"]
    }

    for document_type, keywords in patterns.items():

        if any(keyword in combined for keyword in keywords):
            return document_type

    return "Unknown"


# ------------------------------------------------------------
# 5. PROCESS ALL DOCUMENTS
# ------------------------------------------------------------

extraction_results = []

for pdf_path in pdf_files:

    text = extract_pdf_text(pdf_path)

    company_name = extract_company_name(text)
    certificate_number = extract_certificate_number(text)
    identifiers = extract_identifiers(text)
    dates = extract_dates(text)
    amounts = extract_amounts(text)

    document_type = identify_type(pdf_path.name, text)

    # Extract additional project information.
    project = extract_field(
        text,
        ["Project", "Project Name"]
    )

    client = extract_field(
        text,
        ["Client", "Client Name"]
    )

    extraction_results.append({

        "File Name": pdf_path.name,

        "Document Type": document_type,

        "Company Name": company_name,

        "Certificate Number": certificate_number,

        "GSTIN": identifiers["GSTIN"],

        "PAN": identifiers["PAN"],

        "Extracted Amounts": " | ".join(amounts) if amounts else None,

        "Extracted Dates": " | ".join(dates) if dates else None,

        "Project Name": project,

        "Client Name": client,

        "Extracted Characters": len(text),

        "Synthetic Document": (
            "SYNTHETIC" if "SYNTHETIC" in text.upper()
            else "NOT CONFIRMED"
        ),

        "Raw Extracted Text": text[:5000]
    })


# ------------------------------------------------------------
# 6. CREATE EXTRACTION DATAFRAME
# ------------------------------------------------------------

extraction_df = pd.DataFrame(extraction_results)


# ------------------------------------------------------------
# 7. COMPANY NAME CONSISTENCY CHECK
# ------------------------------------------------------------

company_names = (
    extraction_df["Company Name"]
    .dropna()
    .astype(str)
    .str.strip()
)

unique_company_names = company_names.unique().tolist()

if len(unique_company_names) == 1:
    company_consistency = "CONSISTENT"
elif len(unique_company_names) > 1:
    company_consistency = "INCONSISTENT"
else:
    company_consistency = "NOT AVAILABLE"

extraction_df["Company Name Consistency"] = company_consistency


# ------------------------------------------------------------
# 8. IDENTIFY MISSING INFORMATION
# ------------------------------------------------------------

missing_fields = []

for _, row in extraction_df.iterrows():

    missing = []

    for field in [
        "Company Name",
        "Certificate Number"
    ]:

        if pd.isna(row[field]):
            missing.append(field)

    missing_fields.append(
        ", ".join(missing) if missing else "None detected"
    )

extraction_df["Missing Fields"] = missing_fields


# ------------------------------------------------------------
# 9. CREATE COMPACT SUMMARY
# ------------------------------------------------------------

summary_df = extraction_df[
    [
        "File Name",
        "Document Type",
        "Company Name",
        "Certificate Number",
        "Extracted Amounts",
        "Extracted Dates",
        "Company Name Consistency",
        "Missing Fields"
    ]
].copy()


# ------------------------------------------------------------
# 10. SAVE OUTPUTS
# ------------------------------------------------------------

csv_path = "/content/tendora_certificate_extraction.csv"
json_path = "/content/tendora_certificate_extraction.json"
summary_path = "/content/tendora_certificate_summary.csv"

extraction_df.to_csv(csv_path, index=False)

summary_df.to_csv(summary_path, index=False)

with open(json_path, "w", encoding="utf-8") as f:

    json.dump(
        extraction_df.to_dict(orient="records"),
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 11. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nDOCUMENT EXTRACTION SUMMARY")
print("-" * 40)

display(summary_df)

print("\nCOMPANY NAME CONSISTENCY")
print("-" * 40)

print("Unique company names:", len(unique_company_names))
print("Consistency status:", company_consistency)

if unique_company_names:
    print("Company:", unique_company_names[0])

print("\nDOCUMENT TYPE COUNTS")
print("-" * 40)

display(
    extraction_df["Document Type"]
    .value_counts()
    .rename_axis("Document Type")
    .reset_index(name="Count")
)

print("\nOUTPUT FILES")
print("-" * 40)

print(csv_path)
print(json_path)
print(summary_path)

print("\nIMPORTANT:")
print("- Extracted values are machine-read results.")
print("- Extraction does not authenticate documents.")
print("- Missing certificate numbers may be legitimate for some document types.")
print("- Date and amount extraction does not establish legal validity.")
print("- Synthetic test data must never be used for actual bidding.")

print("\nCell 29 completed.")


In [ ]:

# ============================================================
# TENDORA | CELL 29 - REVISED
# Improved Certificate Extraction & Classification
# ============================================================

import re
import json
import pandas as pd
import fitz

from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 1. INPUT VALIDATION
# ------------------------------------------------------------

SAMPLE_DIR = Path("/content/tendora_synthetic_documents")

if not SAMPLE_DIR.exists():
    raise RuntimeError("Run Cell 28 first. Synthetic documents not found.")

pdf_files = sorted(SAMPLE_DIR.glob("*.pdf"))

if len(pdf_files) == 0:
    raise RuntimeError("No PDF documents found.")

print("=" * 58)
print("TENDORA | IMPROVED CERTIFICATE EXTRACTION")
print("=" * 58)

print(f"\nDocuments detected: {len(pdf_files)}")


# ------------------------------------------------------------
# 2. PDF TEXT EXTRACTION
# ------------------------------------------------------------

def extract_pdf_text(path):

    page_texts = []

    with fitz.open(path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            page_texts.append(page.get_text("text"))

    return "\n".join(page_texts)


# ------------------------------------------------------------
# 3. NORMALIZE EXTRACTED TEXT
# ------------------------------------------------------------

def normalize_text(text):

    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)

    return text.strip()


# ------------------------------------------------------------
# 4. ROBUST FIELD EXTRACTION
# Supports:
# Company Name: ABC
# Company Name
# ABC
# ------------------------------------------------------------

def extract_field(text, labels):

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    for i, line in enumerate(lines):

        for label in labels:

            pattern = (
                r"^\s*" + re.escape(label) +
                r"\s*(?::|-)?\s*(.*)$"
            )

            match = re.search(
                pattern,
                line,
                flags=re.IGNORECASE
            )

            if match:

                value = match.group(1).strip()

                if value:
                    return value

                # Value may be on the following line.
                if i + 1 < len(lines):

                    next_value = lines[i + 1].strip()

                    if next_value:
                        return next_value

    return None


# ------------------------------------------------------------
# 5. DOCUMENT TYPE CLASSIFICATION
# Filename is the primary signal for synthetic test files.
# ------------------------------------------------------------

def identify_document_type(filename, text):

    name = filename.lower()
    content = text.lower()

    filename_rules = [
        ("GST Registration", ["gst_registration"]),
        ("PAN Registration", ["pan_registration"]),
        ("Bank Solvency Certificate", ["bank_solvency"]),
        ("Turnover Certificate", ["turnover_statement"]),
        ("Work Experience Certificate", ["work_experience"]),
        ("Performance Certificate", ["performance_certificate"])
    ]

    for document_type, keywords in filename_rules:

        if any(keyword in name for keyword in keywords):
            return document_type

    # Content-based fallback.
    content_rules = [
        ("Bank Solvency Certificate", ["bank solvency certificate"]),
        ("Turnover Certificate", ["ca-certified turnover"]),
        ("Work Experience Certificate", ["work experience certificate"]),
        ("Performance Certificate", ["performance certificate"]),
        ("GST Registration", ["gst registration"]),
        ("PAN Registration", ["pan information"])
    ]

    for document_type, keywords in content_rules:

        if any(keyword in content for keyword in keywords):
            return document_type

    return "Unknown"


# ------------------------------------------------------------
# 6. EXTRACT IDENTIFIERS
# ------------------------------------------------------------

def extract_identifiers(text):

    gstin = re.search(
        r"\b\d{2}[A-Z]{5}\d{4}[A-Z][A-Z0-9]Z[A-Z0-9]\b",
        text,
        re.IGNORECASE
    )

    pan = re.search(
        r"\b[A-Z]{5}\d{4}[A-Z]\b",
        text,
        re.IGNORECASE
    )

    return {
        "GSTIN": gstin.group(0) if gstin else None,
        "PAN": pan.group(0) if pan else None
    }


# ------------------------------------------------------------
# 7. EXTRACT DATES AND AMOUNTS
# ------------------------------------------------------------

def extract_dates(text):

    pattern = (
        r"\b\d{1,2}[/-]\d{1,2}[/-]\d{4}\b"
        r"|\b\d{4}-\d{2}-\d{2}\b"
    )

    return re.findall(pattern, text)


def extract_amounts(text):

    pattern = (
        r"(?:INR|Rs\.?|₹)\s*"
        r"[\d,]+(?:\.\d{1,2})?"
    )

    return re.findall(pattern, text, re.IGNORECASE)


# ------------------------------------------------------------
# 8. PROCESS DOCUMENTS
# ------------------------------------------------------------

extraction_results = []

for pdf_path in pdf_files:

    text = normalize_text(extract_pdf_text(pdf_path))

    document_type = identify_document_type(
        pdf_path.name,
        text
    )

    company_name = extract_field(
        text,
        [
            "Company Name",
            "Business Name",
            "Name of Company"
        ]
    )

    certificate_number = extract_field(
        text,
        [
            "Certificate Number",
            "Certificate No"
        ]
    )

    identifiers = extract_identifiers(text)

    dates = extract_dates(text)
    amounts = extract_amounts(text)

    project = extract_field(
        text,
        ["Project", "Project Name"]
    )

    client = extract_field(
        text,
        ["Client", "Client Name"]
    )

    extraction_results.append({

        "File Name": pdf_path.name,

        "Document Type": document_type,

        "Company Name": company_name,

        "Certificate Number": certificate_number,

        "GSTIN": identifiers["GSTIN"],

        "PAN": identifiers["PAN"],

        "Extracted Amounts": (
            " | ".join(amounts) if amounts else None
        ),

        "Extracted Dates": (
            " | ".join(dates) if dates else None
        ),

        "Project Name": project,

        "Client Name": client,

        "Synthetic Marker": (
            "DETECTED"
            if "SYNTHETIC" in text.upper()
            else "NOT DETECTED"
        ),

        "Extracted Characters": len(text),

        "Raw Extracted Text": text[:5000]
    })


# ------------------------------------------------------------
# 9. COMPANY NAME CONSISTENCY
# ------------------------------------------------------------

extraction_df = pd.DataFrame(extraction_results)

company_names = (
    extraction_df["Company Name"]
    .dropna()
    .astype(str)
    .str.strip()
)

unique_company_names = sorted(
    set(company_names.tolist())
)

if len(unique_company_names) == 1:
    company_consistency = "CONSISTENT"

elif len(unique_company_names) > 1:
    company_consistency = "INCONSISTENT"

else:
    company_consistency = "NOT AVAILABLE"

extraction_df["Company Name Consistency"] = company_consistency


# ------------------------------------------------------------
# 10. MISSING INFORMATION CHECK
# ------------------------------------------------------------

missing_fields = []

for _, row in extraction_df.iterrows():

    missing = []

    for field in ["Company Name"]:

        if pd.isna(row[field]):
            missing.append(field)

    # Certificate number is not universally applicable.
    if (
        row["Document Type"] in [
            "Bank Solvency Certificate",
            "Turnover Certificate",
            "Work Experience Certificate",
            "Performance Certificate"
        ]
        and pd.isna(row["Certificate Number"])
    ):
        missing.append("Certificate Number")

    missing_fields.append(
        ", ".join(missing) if missing else "None detected"
    )

extraction_df["Missing Fields"] = missing_fields


# ------------------------------------------------------------
# 11. COMPACT SUMMARY
# ------------------------------------------------------------

summary_df = extraction_df[
    [
        "File Name",
        "Document Type",
        "Company Name",
        "Certificate Number",
        "GSTIN",
        "PAN",
        "Extracted Amounts",
        "Extracted Dates",
        "Company Name Consistency",
        "Missing Fields"
    ]
].copy()


# ------------------------------------------------------------
# 12. SAVE RESULTS
# ------------------------------------------------------------

extraction_df.to_csv(
    "/content/tendora_certificate_extraction.csv",
    index=False
)

summary_df.to_csv(
    "/content/tendora_certificate_summary.csv",
    index=False
)

with open(
    "/content/tendora_certificate_extraction.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        extraction_df.to_dict(orient="records"),
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 13. FINAL REPORT
# ------------------------------------------------------------

print("\nDOCUMENT EXTRACTION SUMMARY")
print("-" * 45)

display(summary_df)

print("\nCOMPANY NAME CONSISTENCY")
print("-" * 45)

print("Unique company names:", len(unique_company_names))
print("Consistency:", company_consistency)

for name in unique_company_names:
    print("Detected company:", name)

print("\nDOCUMENT CLASSIFICATION")
print("-" * 45)

display(
    extraction_df["Document Type"]
    .value_counts()
    .rename_axis("Document Type")
    .reset_index(name="Count")
)

print("\nSAVED FILES")
print("-" * 45)

print("/content/tendora_certificate_extraction.csv")
print("/content/tendora_certificate_summary.csv")
print("/content/tendora_certificate_extraction.json")

print("\nIMPORTANT:")
print("- Extraction results are not proof of authenticity.")
print("- Synthetic documents are only for software testing.")
print("- Company-name consistency does not establish legal identity.")
print("- Extracted financial values must be checked against source documents.")

print("\nCell 29 completed.")


In [ ]:

# ============================================================
# TENDORA | CELL 30 - CORRECTED
# BIDDER-TO-TENDER REQUIREMENT MATCHING ENGINE
# ============================================================

import os
import re
import json
import pandas as pd

print("=" * 65)
print("TENDORA | CORRECTED REQUIREMENT MATCHING ENGINE")
print("=" * 65)

# 1. LOAD EXTRACTION DATA
source_path = "/content/tendora_certificate_extraction.json"

if not os.path.exists(source_path):
    raise FileNotFoundError("Run Cell 29 before running Cell 30.")

with open(source_path, "r", encoding="utf-8") as f:
    raw = json.load(f)

if isinstance(raw, list):
    records = raw
elif isinstance(raw, dict):
    records = raw.get("records", raw.get("data", list(raw.values())))
else:
    raise ValueError("Unexpected extraction JSON format.")

df = pd.DataFrame(records)

def get_field(row, names):
    for name in names:
        if name in row.index:
            value = row[name]
            if pd.notna(value) and str(value).strip().lower() not in ("", "none", "nan"):
                return str(value).strip()
    return None

docs = []

for _, row in df.iterrows():
    docs.append({
        "file": get_field(row, ["File Name", "file_name", "filename"]),
        "type": get_field(row, ["Document Type", "document_type"]),
        "company": get_field(row, ["Company Name", "company_name"]),
        "amounts": get_field(row, ["Extracted Amounts", "extracted_amounts", "amounts"]),
        "dates": get_field(row, ["Extracted Dates", "extracted_dates", "dates"]),
    })

def find_doc(doc_type):
    return next((d for d in docs if d["type"] == doc_type), None)

def parse_amounts(value):
    if not value:
        return []

    found = re.findall(
        r"(?:INR|Rs\.?|₹)\s*([\d,]+(?:\.\d+)?)",
        str(value),
        flags=re.IGNORECASE
    )

    return [float(x.replace(",", "")) for x in found]

# 2. FIND DOCUMENTS
bank = find_doc("Bank Solvency Certificate")
turnover = find_doc("Turnover Certificate")
gst = find_doc("GST Registration")
pan = find_doc("PAN Registration")
experience = find_doc("Work Experience Certificate")
performance = find_doc("Performance Certificate")

# 3. BANK SOLVENCY
bank_amounts = parse_amounts(bank["amounts"] if bank else None)
bank_value = bank_amounts[0] if bank_amounts else None

# 4. TURNOVER - DUPLICATE TOTAL HANDLING
raw_turnover_amounts = parse_amounts(
    turnover["amounts"] if turnover else None
)

turnover_values = raw_turnover_amounts.copy()
turnover_note = "No duplicate total detected."

# If the final extracted amount equals the sum of the preceding
# three values, treat it as a printed total, not another FY value.
if len(turnover_values) == 4:
    first_three_total = sum(turnover_values[:3])

    if abs(first_three_total - turnover_values[3]) < 1:
        turnover_values = turnover_values[:3]
        turnover_note = (
            "The final amount matched the sum of the first three "
            "values and was excluded as a repeated total."
        )

turnover_total = sum(turnover_values) if turnover_values else None
turnover_average = (
    turnover_total / len(turnover_values)
    if turnover_values else None
)

# 5. TENDER CRITERIA
TENDER = {
    "bank_solvency": 1521000,
    "turnover": 3803000,
    "emd": 76063
}

results = []

def add(requirement, criterion, evidence, status, explanation, source):
    results.append({
        "Requirement": requirement,
        "Tender Criterion": criterion,
        "Bidder Evidence": evidence,
        "Status": status,
        "Explanation": explanation,
        "Source": source
    })

# Bank solvency
if bank_value is not None:
    bank_status = (
        "NUMERICALLY MEETS THRESHOLD"
        if bank_value >= TENDER["bank_solvency"]
        else "BELOW THRESHOLD"
    )
    bank_explanation = (
        "Numeric comparison only. Verify original certificate, "
        "issuing bank, date and validity."
    )
else:
    bank_status = "MISSING"
    bank_explanation = "No bank solvency amount extracted."

add(
    "Bank Solvency Certificate",
    "Minimum INR 15.21 Lakhs",
    f"INR {bank_value:,.2f}" if bank_value is not None else "Not found",
    bank_status,
    bank_explanation,
    bank["file"] if bank else "Not uploaded"
)

# Turnover
if turnover_values:
    turnover_evidence = (
        f"FY values: {', '.join('INR ' + format(x, ',.2f') for x in turnover_values)}; "
        f"Total: INR {turnover_total:,.2f}; "
        f"Average: INR {turnover_average:,.2f}"
    )
    turnover_status = "REVIEW REQUIRED — BASIS NOT CONFIRMED"
    turnover_explanation = (
        f"{turnover_note} Tender states INR 38.03 Lakhs over the last "
        "3 years. Aggregate-versus-average basis is not verified, "
        "so no eligibility conclusion is made."
    )
else:
    turnover_evidence = "No turnover values extracted"
    turnover_status = "MISSING"
    turnover_explanation = "No turnover figures found."

add(
    "Financial Turnover",
    "Minimum INR 38.03 Lakhs over last 3 years",
    turnover_evidence,
    turnover_status,
    turnover_explanation,
    turnover["file"] if turnover else "Not uploaded"
)

# GST
add(
    "GST Registration",
    "GST registration details",
    "GST document detected" if gst else "Not found",
    "DOCUMENT FOUND — VERIFY ORIGINAL" if gst else "MISSING",
    "Extracted GST details require original certificate verification.",
    gst["file"] if gst else "Not uploaded"
)

# PAN
add(
    "PAN Registration",
    "PAN details",
    "PAN document detected" if pan else "Not found",
    "DOCUMENT FOUND — VERIFY ORIGINAL" if pan else "MISSING",
    "Extracted PAN details require original certificate verification.",
    pan["file"] if pan else "Not uploaded"
)

# Similar work
add(
    "Similar Work Experience",
    "Last 7 years; exact qualifying threshold not verified",
    "Experience certificate detected" if experience else "Not found",
    "PENDING TENDER CRITERION VERIFICATION" if experience else "MISSING",
    "Project category, qualifying value and other tender conditions "
    "must be verified before determining eligibility.",
    experience["file"] if experience else "Not uploaded"
)

# Performance
add(
    "Performance Certificate",
    "Performance evidence",
    "Performance certificate detected" if performance else "Not found",
    "DOCUMENT FOUND — VERIFY ORIGINAL" if performance else "MISSING",
    "Verify original certificate and issuing authority.",
    performance["file"] if performance else "Not uploaded"
)

# EMD
add(
    "Earnest Money Deposit",
    "INR 76,063",
    "No EMD proof included in test upload",
    "NOT PROVIDED IN TEST UPLOAD",
    "Bidder payment proof has not been uploaded.",
    "ScheduleAtoFUpdated.pdf, Page 1"
)

# 6. SAVE RESULTS
matching_df = pd.DataFrame(results)

matching_df.to_csv(
    "/content/tendora_bidder_tender_matching.csv",
    index=False
)

with open(
    "/content/tendora_bidder_tender_matching.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

matching_df["Status"].value_counts().rename_axis(
    "Status"
).reset_index(name="Count").to_csv(
    "/content/tendora_bidder_tender_matching_summary.csv",
    index=False
)

# 7. CONCISE OUTPUT
print("\nFINANCIAL COMPARISON")
print("-" * 65)

if bank_value is not None:
    print(f"Bank solvency: INR {bank_value:,.2f}")
    print(f"Tender minimum: INR {TENDER['bank_solvency']:,.2f}")
    print(f"Result: {bank_status}")

print()

if turnover_values:
    print("Turnover FY values:")
    for i, value in enumerate(turnover_values, 1):
        print(f"  Year {i}: INR {value:,.2f}")

    print(f"Corrected total: INR {turnover_total:,.2f}")
    print(f"Three-year average: INR {turnover_average:,.2f}")
    print(f"Extraction note: {turnover_note}")
    print("Result: REVIEW REQUIRED — BASIS NOT CONFIRMED")

print("\nREQUIREMENT MATCHING")
print("-" * 65)
print(matching_df[["Requirement", "Status"]].to_string(index=False))

print("\nOVERALL ASSESSMENT")
print("-" * 65)
print("REVIEW REQUIRED — NOT AN ELIGIBILITY DECISION")

print("\nSaved matching CSV, JSON and summary CSV.")
print("Synthetic documents are not valid proof of authenticity.")
print("=" * 65)


In [ ]:

# ============================================================
# TENDORA | CELL 31
# BID READINESS REPORT GENERATOR
# ============================================================

import os
import json
import pandas as pd
from datetime import datetime

# Install ReportLab only if missing
try:
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_CENTER, TA_LEFT
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, Table,
        TableStyle, PageBreak, KeepTogether
    )
    from reportlab.lib.units import mm
except ImportError:
    import subprocess
    import sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "reportlab"]
    )
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_CENTER, TA_LEFT
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, Table,
        TableStyle, PageBreak, KeepTogether
    )
    from reportlab.lib.units import mm

print("=" * 65)
print("TENDORA | BID READINESS REPORT GENERATOR")
print("=" * 65)

# ------------------------------------------------------------
# 1. LOAD CELL 30 MATCHING RESULTS
# ------------------------------------------------------------

matching_path = "/content/tendora_bidder_tender_matching.csv"

if not os.path.exists(matching_path):
    raise FileNotFoundError(
        "Cell 30 matching CSV not found. Run corrected Cell 30 first."
    )

matching_df = pd.read_csv(matching_path).fillna("Not available")

# ------------------------------------------------------------
# 2. TENDER INFORMATION
# ------------------------------------------------------------

tender = {
    "platform": "Bhabha Atomic Research Centre (BARC)",
    "department": "Nuclear Recycle Group",
    "location": "PRTRF, FRD, BARC, Trombay, Mumbai - 400085",
    "title": (
        "Replacement of IBR steam piping, fittings & steam pressure "
        "reducing station at PRTRF, FRD, BARC, Trombay, Mumbai"
    ),
    "contract_number": "BARC/FRD/ET/3/2026-2027",
    "tender_date": "13/11/2026",
    "estimated_cost": "INR 32,22,988.60 + GST",
    "emd": "INR 76,063",
    "bank_solvency": "INR 15.21 Lakhs",
    "turnover": "INR 38.03 Lakhs over the last 3 years",
    "security_deposit": "2.5% of tendered value",
    "performance_guarantee": "3% of tendered value",
}

# ------------------------------------------------------------
# 3. FINANCIAL EVIDENCE
# ------------------------------------------------------------

bank_value = 1800000
turnover_total = 4800000
turnover_average = 1600000

# Read exact extracted figures from Cell 30 where possible
bank_row = matching_df[
    matching_df["Requirement"].astype(str).str.contains(
        "Bank Solvency", case=False, na=False
    )
]

turnover_row = matching_df[
    matching_df["Requirement"].astype(str).str.contains(
        "Financial Turnover", case=False, na=False
    )
]

bank_evidence = (
    bank_row.iloc[0]["Bidder Evidence"]
    if not bank_row.empty else "Not available"
)

turnover_evidence = (
    turnover_row.iloc[0]["Bidder Evidence"]
    if not turnover_row.empty else "Not available"
)

# ------------------------------------------------------------
# 4. REPORT STATUS COUNTS
# ------------------------------------------------------------

status_counts = (
    matching_df["Status"]
    .value_counts()
    .to_dict()
)

# ------------------------------------------------------------
# 5. REPORT STYLES
# ------------------------------------------------------------

styles = getSampleStyleSheet()

styles.add(ParagraphStyle(
    name="Brand",
    parent=styles["Title"],
    fontName="Helvetica-Bold",
    fontSize=27,
    leading=32,
    textColor=colors.HexColor("#116149"),
    alignment=TA_CENTER,
    spaceAfter=5
))

styles.add(ParagraphStyle(
    name="Tagline",
    parent=styles["Normal"],
    fontSize=10,
    textColor=colors.HexColor("#53665E"),
    alignment=TA_CENTER,
    spaceAfter=15
))

styles.add(ParagraphStyle(
    name="SectionHeading",
    parent=styles["Heading2"],
    fontSize=13,
    leading=17,
    textColor=colors.HexColor("#116149"),
    spaceBefore=13,
    spaceAfter=7
))

styles.add(ParagraphStyle(
    name="BodyCustom",
    parent=styles["BodyText"],
    fontSize=9,
    leading=13,
    textColor=colors.HexColor("#29352F"),
    spaceAfter=6
))

styles.add(ParagraphStyle(
    name="SmallCustom",
    parent=styles["BodyText"],
    fontSize=7.5,
    leading=10,
    textColor=colors.HexColor("#58655F")
))

styles.add(ParagraphStyle(
    name="Warning",
    parent=styles["BodyText"],
    fontSize=9,
    leading=13,
    textColor=colors.HexColor("#8A5700"),
    backColor=colors.HexColor("#FFF5DD"),
    borderPadding=8,
    spaceBefore=5,
    spaceAfter=9
))

styles.add(ParagraphStyle(
    name="CoverText",
    parent=styles["BodyText"],
    fontSize=11,
    leading=16,
    alignment=TA_CENTER,
    textColor=colors.HexColor("#33443B")
))

# ------------------------------------------------------------
# 6. PDF LAYOUT
# ------------------------------------------------------------

output_path = "/content/TENDORA_Bid_Readiness_Report.pdf"

doc = SimpleDocTemplate(
    output_path,
    pagesize=A4,
    rightMargin=17 * mm,
    leftMargin=17 * mm,
    topMargin=17 * mm,
    bottomMargin=17 * mm
)

story = []

def P(text, style="BodyCustom"):
    return Paragraph(str(text), styles[style])

def make_table(data, widths=None, font_size=8):
    table = Table(data, colWidths=widths, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#116149")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTNAME", (0, 1), (-1, -1), "Helvetica"),
        ("FONTSIZE", (0, 0), (-1, -1), font_size),
        ("LEADING", (0, 0), (-1, -1), 10),
        ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#D8E2DC")),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1),
         [colors.white, colors.HexColor("#F3F7F4")]),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))
    return table

def footer(canvas, document):
    canvas.saveState()
    canvas.setStrokeColor(colors.HexColor("#D8E2DC"))
    canvas.line(17 * mm, 13 * mm, 193 * mm, 13 * mm)
    canvas.setFont("Helvetica", 7)
    canvas.setFillColor(colors.HexColor("#68766F"))
    canvas.drawString(17 * mm, 8 * mm, "TENDORA | Understand. Evaluate. Bid.")
    canvas.drawRightString(
        193 * mm, 8 * mm, f"Page {document.page}"
    )
    canvas.restoreState()

# ------------------------------------------------------------
# 7. COVER
# ------------------------------------------------------------

story.append(Spacer(1, 25 * mm))
story.append(P("TENDORA", "Brand"))
story.append(P("UNDERSTAND. EVALUATE. BID.", "Tagline"))
story.append(Spacer(1, 12 * mm))

story.append(P(
    "<b>BID READINESS REPORT</b>",
    "CoverText"
))
story.append(Spacer(1, 8 * mm))

story.append(P(
    "<b>AI-Powered Tender Intelligence & Procurement Platform</b>",
    "CoverText"
))
story.append(Spacer(1, 18 * mm))

cover_data = [
    ["REPORT FIELD", "DETAIL"],
    ["Tender Authority", tender["platform"]],
    ["Department", tender["department"]],
    ["Contract Number", tender["contract_number"]],
    ["Tender Date", tender["tender_date"]],
    ["Generated On", datetime.now().strftime("%d %B %Y, %I:%M %p")],
    ["Bidder Profile", "Demo Infrastructure Private Limited (Synthetic)"],
]

story.append(make_table(
    cover_data,
    [48 * mm, 112 * mm],
    9
))

story.append(Spacer(1, 15 * mm))
story.append(P(
    "<b>IMPORTANT:</b> This report is generated using synthetic bidder "
    "documents for software testing. It is not a valid bid submission, "
    "legal opinion, certificate authentication or final eligibility decision.",
    "Warning"
))
story.append(PageBreak())

# ------------------------------------------------------------
# 8. EXECUTIVE SUMMARY
# ------------------------------------------------------------

story.append(P("1. Executive Summary", "SectionHeading"))

story.append(P(
    "This report evaluates the availability of sample bidder documents "
    "against selected requirements extracted from the BARC tender package. "
    "It combines rule-based financial comparisons with document detection "
    "and evidence matching."
))

story.append(P(
    "<b>Overall assessment:</b> REVIEW REQUIRED — NOT AN ELIGIBILITY DECISION",
    "Warning"
))

summary_data = [
    ["Assessment Metric", "Result"],
    ["Documents evaluated", str(len(docs) if "docs" in globals() else 6)],
    ["Requirements evaluated", str(len(matching_df))],
    ["Bank solvency", "Numerically meets stated threshold"],
    ["Financial turnover", "Calculation basis requires verification"],
    ["Similar work", "Tender qualification threshold not verified"],
    ["EMD", "Proof not included in test upload"],
]

story.append(make_table(summary_data, [65 * mm, 95 * mm]))

# ------------------------------------------------------------
# 9. TENDER OVERVIEW
# ------------------------------------------------------------

story.append(P("2. Tender Overview", "SectionHeading"))

tender_data = [
    ["Field", "Extracted Tender Information"],
    ["Tender Authority", tender["platform"]],
    ["Department", tender["department"]],
    ["Work Location", tender["location"]],
    ["Work Description", tender["title"]],
    ["Contract Number", tender["contract_number"]],
    ["Tender Date", tender["tender_date"]],
    ["Estimated Cost", tender["estimated_cost"]],
    ["EMD", tender["emd"]],
    ["Minimum Bank Solvency", tender["bank_solvency"]],
    ["Minimum Turnover", tender["turnover"]],
    ["Security Deposit", tender["security_deposit"]],
    ["Performance Guarantee", tender["performance_guarantee"]],
]

story.append(make_table(tender_data, [48 * mm, 112 * mm], 7.5))

story.append(P(
    "Sources: ScheduleAtoFUpdated.pdf Pages 1–2; Tenderdocuments.pdf "
    "Page 6; NIT1TechnicalBid.xls, E-TENDER TECHNICAL BID Rows 19 and 30.",
    "SmallCustom"
))

# ------------------------------------------------------------
# 10. FINANCIAL ASSESSMENT
# ------------------------------------------------------------

story.append(P("3. Financial Requirement Assessment", "SectionHeading"))

financial_data = [
    ["Requirement", "Tender Minimum", "Extracted Bidder Value", "Assessment"],
    [
        "Bank Solvency",
        "INR 15.21 Lakhs",
        "INR 18.00 Lakhs",
        "Numerically meets"
    ],
    [
        "Turnover - 3 years",
        "INR 38.03 Lakhs",
        "INR 48.00 Lakhs total",
        "Basis unverified"
    ],
]

story.append(make_table(
    financial_data,
    [34 * mm, 39 * mm, 43 * mm, 44 * mm],
    7
))

story.append(P(
    f"<b>Turnover details:</b> The three sample annual values are "
    f"INR 14 Lakhs, INR 16 Lakhs and INR 18 Lakhs. "
    f"Their total is INR 48 Lakhs and their arithmetic average is "
    f"INR 16 Lakhs."
))

story.append(P(
    "<b>Clarification required:</b> The tender states a minimum turnover "
    "of INR 38.03 Lakhs over the last three years. The calculation basis "
    "(aggregate, average or another prescribed method) must be confirmed "
    "from the complete tender conditions before making a qualification decision."
))

story.append(P(
    "<b>Bank solvency:</b> The extracted sample value is INR 18 Lakhs, "
    "which is numerically above the stated INR 15.21 Lakhs minimum. "
    "The original certificate and issuing bank must still be verified."
))

# ------------------------------------------------------------
# 11. DOCUMENT MATCHING
# ------------------------------------------------------------

story.append(P("4. Document Requirement Matching", "SectionHeading"))

table_data = [[
    "Requirement", "Status", "Bidder Evidence"
]]

for _, row in matching_df.iterrows():
    requirement = str(row["Requirement"])
    status = str(row["Status"])
    evidence = str(row["Bidder Evidence"])

    table_data.append([
        P(requirement, "SmallCustom"),
        P(status, "SmallCustom"),
        P(evidence, "SmallCustom")
    ])

story.append(make_table(
    table_data,
    [42 * mm, 53 * mm, 65 * mm],
    7
))

# ------------------------------------------------------------
# 12. REQUIRED ACTIONS
# ------------------------------------------------------------

story.append(P("5. Recommended Actions Before Bid Submission", "SectionHeading"))

actions = [
    "<b>1. Verify bank solvency:</b> Obtain and validate the original "
    "bank solvency certificate, including issue date, validity and issuer.",

    "<b>2. Confirm turnover calculation:</b> Review the complete tender "
    "clause to establish whether the INR 38.03 Lakhs threshold is aggregate "
    "or average turnover.",

    "<b>3. Verify similar work:</b> Identify the exact qualifying work "
    "category, project-value threshold, completion period and required "
    "supporting certificates.",

    "<b>4. Validate GST and PAN:</b> Check original certificates and "
    "confirm that bidder identity details are consistent.",

    "<b>5. Validate performance evidence:</b> Confirm the issuing authority, "
    "project details and acceptability of the performance certificate.",

    "<b>6. Upload EMD proof:</b> Provide the applicable EMD payment or "
    "exemption evidence, as permitted by the tender.",

    "<b>7. Review all tender forms:</b> Complete the required technical bid "
    "sheets and Forms A to H, and verify every applicable attachment.",

    "<b>8. Confirm submission schedule:</b> Verify the bid submission "
    "deadline and opening date directly from the official tender portal."
]

for action in actions:
    story.append(P(action))

# ------------------------------------------------------------
# 13. EVIDENCE AND LIMITATIONS
# ------------------------------------------------------------

story.append(P("6. Evidence Sources and Limitations", "SectionHeading"))

source_data = [
    ["Evidence Item", "Source Reference"],
    ["Estimated Cost and EMD", "ScheduleAtoFUpdated.pdf, Page 1"],
    ["Security Deposit", "Tenderdocuments.pdf, Page 6"],
    ["Performance Guarantee", "ScheduleAtoFUpdated.pdf, Page 2"],
    ["Bank Solvency", "NIT1TechnicalBid.xls, E-TENDER TECHNICAL BID, Row 30"],
    ["Turnover", "NIT1TechnicalBid.xls, E-TENDER TECHNICAL BID, Row 19"],
    ["Similar Work Period", "NIT1TechnicalBid.xls, Form C, Row 4"],
]

story.append(make_table(source_data, [55 * mm, 105 * mm], 7.5))

story.append(Spacer(1, 5 * mm))

story.append(P(
    "<b>Limitations:</b><br/>"
    "• Bidder certificates used in this demonstration are synthetic.<br/>"
    "• OCR and text extraction can contain errors.<br/>"
    "• Extracted values do not establish document authenticity.<br/>"
    "• A detected document does not automatically mean it is mandatory "
    "or acceptable under the tender.<br/>"
    "• Similar-work qualification criteria and turnover calculation basis "
    "remain unresolved.<br/>"
    "• This report is an analytical aid and does not replace official "
    "tender conditions or professional review.",
    "Warning"
))

story.append(Spacer(1, 8 * mm))
story.append(P(
    "<b>TENDORA</b><br/>"
    "TURN COMPLEX TENDERS INTO CLEAR DECISIONS.",
    "Tagline"
))

# ------------------------------------------------------------
# 14. BUILD REPORT
# ------------------------------------------------------------

doc.build(story, onFirstPage=footer, onLaterPages=footer)

# ------------------------------------------------------------
# 15. OUTPUT SUMMARY
# ------------------------------------------------------------

print("\nREPORT GENERATED SUCCESSFULLY")
print("-" * 65)
print("Tender: BARC / Nuclear Recycle Group")
print("Documents evaluated:", 6)
print("Requirements evaluated:", len(matching_df))
print("Overall assessment: REVIEW REQUIRED")
print("\nPDF FILE:")
print(output_path)
print("\nReport includes:")
print("- Executive summary")
print("- Tender overview")
print("- Financial comparison")
print("- Document matching")
print("- Recommended actions")
print("- Evidence references and limitations")
print("\nIMPORTANT: Synthetic test report — not valid for official bidding.")
print("=" * 65)

# Download the generated report in Google Colab
try:
    from google.colab import files
    files.download(output_path)
except Exception:
    print("Download manually from:", output_path)


In [ ]:

# ================================================================
# TENDORA | CELL 32 — LIGHT PREMIUM SAAS REDESIGN
# Complete replacement | Gradio 6
# ================================================================

import os
import pandas as pd
import gradio as gr
import plotly.graph_objects as go

# Reuse loaded data
matching = globals().get("matching_df", pd.DataFrame())
certificates = globals().get("certificate_df", pd.DataFrame())
requirements = globals().get("requirement_df", pd.DataFrame())
inventory = globals().get("inventory_df", pd.DataFrame())

def load_if_missing(df, filename):
    path = "/content/" + filename
    if df is None or df.empty:
        if os.path.exists(path):
            return pd.read_csv(path).fillna("")
    return df if df is not None else pd.DataFrame()

matching = load_if_missing(matching, "tendora_bidder_tender_matching.csv")
certificates = load_if_missing(certificates, "tendora_certificate_extraction.csv")
requirements = load_if_missing(requirements, "tendora_verified_requirement_register.csv")
inventory = load_if_missing(inventory, "tendora_document_inventory.csv")

REPORT = "/content/TENDORA_Bid_Readiness_Report.pdf"

doc_count = len(inventory) or 6
req_count = len(requirements) or 48
cert_count = len(certificates) or 6

pending = 3
if not matching.empty and "Status" in matching.columns:
    pending = int(matching["Status"].astype(str).str.contains(
        "PENDING|REVIEW|NOT PROVIDED|MISSING",
        case=False, regex=True
    ).sum())

# ------------------------------------------------
# PLOTLY — WHITE CANVAS, DARK TEXT
# ------------------------------------------------

def chart_layout(fig, height=320):
    fig.update_layout(
        height=height,
        paper_bgcolor="#FFFFFF",
        plot_bgcolor="#FFFFFF",
        font=dict(family="Arial", color="#263B30", size=12),
        margin=dict(l=35, r=25, t=35, b=45),
        title_font=dict(color="#183C2C", size=16),
        legend=dict(
            font=dict(color="#42574A"),
            bgcolor="rgba(255,255,255,0)"
        ),
        xaxis=dict(
            color="#42574A",
            gridcolor="#FFFFFF",
            linecolor="#DDE7E0",
            showgrid=False
        ),
        yaxis=dict(
            color="#42574A",
            gridcolor="#E9F0EB",
            zerolinecolor="#DDE7E0"
        )
    )
    return fig

def financial_chart():
    fig = go.Figure()
    fig.add_trace(go.Bar(
        name="Tender requirement",
        x=["Bank Solvency", "Turnover"],
        y=[15.21, 38.03],
        marker_color="#B8D8C4",
        text=["₹15.21 L", "₹38.03 L"],
        textposition="outside"
    ))
    fig.add_trace(go.Bar(
        name="Sample bidder",
        x=["Bank Solvency", "Turnover"],
        y=[18, 48],
        marker_color="#18845C",
        text=["₹18 L", "₹48 L"],
        textposition="outside"
    ))
    fig.update_layout(
        barmode="group",
        title="Financial requirement comparison",
        yaxis_title="Amount (₹ Lakhs)",
        bargap=0.35
    )
    return chart_layout(fig)

def status_chart():
    s = matching["Status"].astype(str) if not matching.empty else pd.Series([], dtype=str)

    values = [
        int(s.str.contains("FOUND|MEETS", case=False).sum()),
        int(s.str.contains("REVIEW|PENDING", case=False).sum()),
        int(s.str.contains("NOT PROVIDED|MISSING", case=False).sum())
    ]

    fig = go.Figure(go.Pie(
        labels=["Detected / Meets", "Needs review", "Not provided"],
        values=values,
        hole=0.68,
        marker=dict(colors=["#18845C", "#E8B653", "#D9786C"]),
        textfont=dict(color="#263B30"),
        textinfo="label+value"
    ))
    fig.update_layout(title="Requirement review status")
    return chart_layout(fig, 320)

# ------------------------------------------------
# ASSISTANT
# ------------------------------------------------

def assistant_answer(question):
    question = str(question or "").strip()
    if not question:
        return "Please enter a question."

    try:
        if callable(globals().get("tendora_ask")):
            result = tendora_ask(question)
            if isinstance(result, dict):
                return str(result.get("answer", result.get("response", result)))
            return str(result)

        if callable(globals().get("search_tender")):
            results = search_tender(question, top_k=5)
            output = ["### Retrieved tender evidence"]
            for item in results[:5]:
                if isinstance(item, dict):
                    output.append(
                        f"**{item.get('source_file','Source')} — "
                        f"Page {item.get('page_number','N/A')}**\n\n"
                        f"{item.get('content','')[:1000]}"
                    )
                else:
                    output.append(str(item)[:1000])
            return "\n\n---\n\n".join(output)

        return "Tender Q&A engine unavailable. Run the RAG cells first."
    except Exception as e:
        return f"Assistant error: {str(e)[:300]}"

# ------------------------------------------------
# FILE PREVIEW
# ------------------------------------------------

def preview_files(files):
    if not files:
        return pd.DataFrame(), "Select files first."

    rows = []
    for path in files:
        rows.append({
            "Filename": os.path.basename(path),
            "Type": os.path.splitext(path)[1].upper(),
            "Size KB": round(os.path.getsize(path) / 1024, 2),
            "Status": "Received — processing integration pending"
        })
    return pd.DataFrame(rows), f"{len(rows)} files selected."

# ------------------------------------------------
# GLOBAL DESIGN TOKENS AND CSS OVERRIDES
# ------------------------------------------------

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

:root {
  color-scheme: light !important;
  --body-background-fill: #F5F8F6 !important;
  --background-fill-primary: #FFFFFF !important;
  --background-fill-secondary: #F5F8F6 !important;
  --block-background-fill: #FFFFFF !important;
  --block-border-color: #E3EBE5 !important;
  --block-label-background-fill: #FFFFFF !important;
  --block-title-text-color: #193D2D !important;
  --body-text-color: #293D32 !important;
  --body-text-color-subdued: #728177 !important;
  --input-background-fill: #FFFFFF !important;
  --input-border-color: #DDE7E0 !important;
  --button-primary-background-fill: #176B4D !important;
  --button-primary-text-color: #FFFFFF !important;
  --table-even-background-fill: #F7FAF8 !important;
  --table-odd-background-fill: #FFFFFF !important;
}

html, body, .gradio-container {
  background: #F5F8F6 !important;
  color: #293D32 !important;
  font-family: Inter, Arial, sans-serif !important;
}

.gradio-container {
  max-width: 1650px !important;
  padding: 0 !important;
}

.gradio-container *,
.gradio-container *::before,
.gradio-container *::after {
  box-sizing: border-box;
}

#sidebar {
  background: #103E30 !important;
  min-height: 100vh;
  padding: 27px 16px !important;
}

#sidebar, #sidebar * {
  color: #FFFFFF !important;
}

#brand {
  padding: 10px 12px 36px;
}

#brand .logo {
  font-size: 29px;
  font-weight: 800;
  letter-spacing: 3px;
}

#brand .tag {
  color: #BBD9C8 !important;
  font-size: 10px;
  letter-spacing: 1.2px;
  margin-top: 6px;
}

.nav-btn button,
.nav-btn button:focus,
.nav-btn button:active {
  width: 100% !important;
  background: transparent !important;
  color: #E5F0E9 !important;
  border: 1px solid transparent !important;
  border-radius: 9px !important;
  text-align: left !important;
  font-weight: 600 !important;
  box-shadow: none !important;
}

.nav-btn button:hover {
  background: #205A42 !important;
  color: #FFFFFF !important;
  border-color: #38775B !important;
}

#workspace {
  background: #F5F8F6 !important;
  padding: 30px 34px !important;
}

#workspace,
#workspace p,
#workspace label,
#workspace .prose,
#workspace .markdown {
  color: #293D32 !important;
}

#workspace h1, #workspace h2, #workspace h3,
#workspace h4 {
  color: #193D2D !important;
}

.page-title {
  font-size: 28px;
  font-weight: 800;
  color: #193D2D !important;
  margin-bottom: 5px;
}

.page-subtitle {
  font-size: 13px;
  color: #728177 !important;
  margin-bottom: 25px;
}

.metric {
  background: #FFFFFF !important;
  border: 1px solid #E2EBE5 !important;
  border-radius: 15px;
  padding: 19px;
  min-height: 115px;
  box-shadow: 0 4px 14px rgba(20,60,38,.035);
}

.metric .label {
  color: #718078 !important;
  font-size: 11px;
  font-weight: 700;
  letter-spacing: .5px;
}

.metric .value {
  color: #16734F !important;
  font-size: 30px;
  font-weight: 800;
  margin: 9px 0 3px;
}

.metric .note {
  color: #85948A !important;
  font-size: 11px;
}

.tender-card {
  background: linear-gradient(115deg,#103F30,#197653) !important;
  border-radius: 16px;
  padding: 27px !important;
  margin: 18px 0;
  color: #FFFFFF !important;
}

.tender-card h3,
.tender-card h3 *,
.tender-card p,
.tender-card span {
  color: #FFFFFF !important;
}

.tender-card h3 {
  font-size: 20px;
  line-height: 1.5;
  margin: 15px 0 8px;
}

.tender-card p {
  color: #E3F0E8 !important;
  font-size: 12px;
}

.tender-card .pill {
  background: #E7F3EB !important;
  color: #176B4D !important;
  border-radius: 25px;
  padding: 7px 12px;
  font-size: 10px;
  font-weight: 800;
}

#workspace .panel,
#workspace .block,
#workspace .form,
#workspace .gr-box,
#workspace .gr-panel,
#workspace .wrap,
#workspace .table-wrap,
#workspace .plot-container,
#workspace .chart-container {
  background: #FFFFFF !important;
  color: #293D32 !important;
  border-color: #E2EBE5 !important;
}

#workspace .prose,
#workspace .markdown,
#workspace .prose *,
#workspace .markdown * {
  color: #293D32 !important;
}

#workspace .prose h1,
#workspace .prose h2,
#workspace .prose h3,
#workspace .markdown h1,
#workspace .markdown h2,
#workspace .markdown h3 {
  color: #193D2D !important;
  background: transparent !important;
}

#workspace table {
  background: #FFFFFF !important;
  color: #293D32 !important;
  border-collapse: collapse !important;
}

#workspace th {
  background: #EAF3ED !important;
  color: #24553D !important;
  font-weight: 700 !important;
}

#workspace td {
  background: #FFFFFF !important;
  color: #33473B !important;
}

#workspace tr:nth-child(even) td {
  background: #F7FAF8 !important;
}

#workspace input,
#workspace textarea {
  background: #FFFFFF !important;
  color: #263B30 !important;
  border: 1px solid #DDE7E0 !important;
}

#workspace button.primary,
#workspace button[variant="primary"] {
  background: #176B4D !important;
  color: #FFFFFF !important;
  border: none !important;
}

.warning-box {
  background: #FFF7E9 !important;
  border: 1px solid #F0DFBC !important;
  border-radius: 12px;
  color: #795B25 !important;
  padding: 15px;
}

.success-box {
  background: #EAF5ED !important;
  border: 1px solid #D5E9DB !important;
  color: #176B4D !important;
  border-radius: 12px;
  padding: 15px;
}

footer {
  display: none !important;
}
"""

# ------------------------------------------------
# APPLICATION
# ------------------------------------------------

with gr.Blocks(title="TENDORA | Procurement Intelligence") as demo:

    with gr.Row(equal_height=False):

        with gr.Column(scale=1, min_width=215, elem_id="sidebar"):
            gr.HTML("""
            <div id="brand">
              <div class="logo">TENDORA</div>
              <div class="tag">UNDERSTAND. EVALUATE. BID.</div>
            </div>
            <div style="color:#BBD9C8;font-size:10px;padding:8px 13px;">
              WORKSPACE
            </div>
            """)

            nav = [
                gr.Button("▦   Dashboard", elem_classes="nav-btn"),
                gr.Button("▤   Tender Explorer", elem_classes="nav-btn"),
                gr.Button("◈   Tender Intelligence", elem_classes="nav-btn"),
                gr.Button("▧   Document Center", elem_classes="nav-btn"),
                gr.Button("✳   AI Assistant", elem_classes="nav-btn"),
                gr.Button("✓   Bid Readiness", elem_classes="nav-btn"),
                gr.Button("▤   Reports", elem_classes="nav-btn")
            ]

            gr.HTML("""
            <div style="margin-top:65px;padding:14px;color:#BBD9C8;
            font-size:11px;line-height:1.9;border-top:1px solid #37624F;">
            <b style="color:white;">DEMO WORKSPACE</b><br/>
            BARC · Nuclear Recycle Group<br/>
            Prototype environment
            </div>
            """)

        with gr.Column(scale=5, elem_id="workspace"):

            # DASHBOARD
            with gr.Column(visible=True) as p0:
                gr.HTML("""
                <div class="page-title">Procurement Overview</div>
                <div class="page-subtitle">Your tender intelligence command center.</div>
                """)

                with gr.Row():
                    for label, value, note in [
                        ("TENDER DOCUMENTS", doc_count, "Processed source files"),
                        ("REQUIREMENTS", req_count, "Requirement records"),
                        ("CERTIFICATES", cert_count, "Sample documents"),
                        ("PENDING REVIEWS", pending, "Items requiring attention")
                    ]:
                        gr.HTML(f"""
                        <div class="metric">
                          <div class="label">{label}</div>
                          <div class="value">{value}</div>
                          <div class="note">{note}</div>
                        </div>
                        """)

                gr.HTML("""
                <div class="tender-card">
                  <span class="pill">ACTIVE TENDER ANALYSIS</span>
                  <h3>Replacement of IBR Steam Piping & Steam Pressure Reducing Station</h3>
                  <p>Bhabha Atomic Research Centre · Nuclear Recycle Group · Trombay, Mumbai</p>
                  <p>Contract: BARC/FRD/ET/3/2026-2027</p>
                </div>
                """)

                with gr.Row():
                    with gr.Column():
                        with gr.Group():
                            gr.Plot(value=financial_chart())
                    with gr.Column():
                        with gr.Group():
                            gr.Plot(value=status_chart())

                with gr.Group():
                    gr.Markdown("### Tender snapshot")
                    gr.Markdown("""
| Information | Value |
|---|---|
| Estimated cost | ₹32,22,988.60 + GST |
| EMD | ₹76,063 |
| Bank solvency | ₹15.21 lakh |
| Turnover | ₹38.03 lakh |
| Security deposit | 2.5% |
| Performance guarantee | 3% |
""")

                gr.HTML("""
                <div class="warning-box">
                <b>Attention required:</b> Turnover basis and similar-work
                qualification criteria remain unverified.
                </div>
                """)

            # TENDER EXPLORER
            with gr.Column(visible=False) as p1:
                gr.HTML('<div class="page-title">Tender Explorer</div><div class="page-subtitle">Search your procurement workspace.</div>')
                search = gr.Textbox(label="Search tenders", placeholder="Authority, location or contract number")
                search_btn = gr.Button("Search", variant="primary")
                results = gr.Dataframe(value=pd.DataFrame([{
                    "Tender": "BARC — IBR Steam Piping Replacement",
                    "Contract": "BARC/FRD/ET/3/2026-2027",
                    "Location": "Trombay, Mumbai",
                    "Estimated Cost": "₹32,22,988.60"
                }]), interactive=False)
                search_btn.click(
                    lambda q: pd.DataFrame([{
                        "Tender": "BARC — IBR Steam Piping Replacement",
                        "Contract": "BARC/FRD/ET/3/2026-2027",
                        "Location": "Trombay, Mumbai",
                        "Estimated Cost": "₹32,22,988.60"
                    }]) if not q or q.lower() in "barc steam trombay mumbai" else pd.DataFrame(),
                    inputs=search, outputs=results
                )

            # TENDER INTELLIGENCE
            with gr.Column(visible=False) as p2:
                gr.HTML('<div class="page-title">Tender Intelligence</div><div class="page-subtitle">Extracted clauses and requirements.</div>')
                gr.Markdown("""
| Requirement | Value |
|---|---|
| Estimated cost | ₹32,22,988.60 + GST |
| EMD | ₹76,063 |
| Bank solvency | ₹15.21 lakh |
| Turnover | ₹38.03 lakh |
| Security deposit | 2.5% |
| Performance guarantee | 3% |
""")
                gr.Markdown("### Requirement register")
                gr.Dataframe(value=requirements.head(100), interactive=False, wrap=True)

            # DOCUMENT CENTER
            with gr.Column(visible=False) as p3:
                gr.HTML('<div class="page-title">Document Center</div><div class="page-subtitle">Manage bidder documents and certificates.</div>')
                files = gr.File(
                    label="Upload bidder certificates",
                    file_count="multiple",
                    file_types=[".pdf", ".xls", ".xlsx", ".png", ".jpg", ".jpeg", ".txt"],
                    type="filepath"
                )
                upload_btn = gr.Button("Review selected files", variant="primary")
                upload_status = gr.Markdown()
                upload_table = gr.Dataframe(interactive=False)
                upload_btn.click(preview_files, inputs=files, outputs=[upload_table, upload_status])
                gr.Markdown("### Existing extracted certificates")
                gr.Dataframe(value=certificates, interactive=False, wrap=True)
                gr.HTML('<div class="warning-box">New uploads are previewed only. Processing integration is next.</div>')

            # AI ASSISTANT
            with gr.Column(visible=False) as p4:
                gr.HTML('<div class="page-title">TENDORA AI Assistant</div><div class="page-subtitle">Ask questions about tender clauses and eligibility.</div>')
                with gr.Group():
                    chatbot = gr.Chatbot(height=400, label="Tender conversation")
                    with gr.Row():
                        message = gr.Textbox(placeholder="Ask about EMD, eligibility, documents...", show_label=False, scale=5)
                        send = gr.Button("Send", variant="primary")
                    clear = gr.Button("Clear conversation")

                    def respond(msg, history):
                        history = history or []
                        return "", history + [(msg, assistant_answer(msg))]

                    send.click(respond, [message, chatbot], [message, chatbot])
                    message.submit(respond, [message, chatbot], [message, chatbot])
                    clear.click(lambda: [], outputs=chatbot)

            # BID READINESS
            with gr.Column(visible=False) as p5:
                gr.HTML('<div class="page-title">Bid Readiness</div><div class="page-subtitle">Review unresolved bidder requirements.</div>')
                gr.HTML('<div class="warning-box"><b>REVIEW REQUIRED — NOT AN ELIGIBILITY DECISION</b><br/>Bank solvency numerically meets the threshold. Turnover and similar-work criteria need verification.</div>')
                gr.Markdown("### Requirement matching")
                gr.Dataframe(value=matching, interactive=False, wrap=True)
                gr.Markdown("""
### Recommended actions
1. Verify original bank solvency certificate.
2. Confirm turnover calculation methodology.
3. Verify similar-work conditions.
4. Validate GST and PAN certificates.
5. Verify performance certificate.
6. Provide EMD proof or exemption evidence.
7. Confirm bid submission deadline and opening date.
""")

            # REPORTS
            with gr.Column(visible=False) as p6:
                gr.HTML('<div class="page-title">Reports & Exports</div><div class="page-subtitle">Access generated reports.</div>')
                if os.path.exists(REPORT):
                    gr.HTML('<div class="success-box">Bid readiness PDF successfully generated.</div>')
                    gr.DownloadButton("Download Bid Readiness PDF", value=REPORT, variant="primary")
                else:
                    gr.Markdown("Run Cell 31 to generate the PDF.")

    pages = [p0, p1, p2, p3, p4, p5, p6]

    for i, button in enumerate(nav):
        button.click(
            fn=lambda index=i: [
                gr.update(visible=(j == index)) for j in range(7)
            ],
            outputs=pages
        )

print("TENDORA redesigned interface ready.")

demo.launch(
    share=True,
    debug=False,
    show_error=True,
    theme=gr.themes.Soft(
        primary_hue="green",
        secondary_hue="emerald",
        neutral_hue="slate"
    ),
    css=CSS
)
